# 1. EDA


### Import thư viện


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
plt.style.use('seaborn-v0_8-whitegrid')

## 1.1. Load data và xem nhanh dữ liệu
Load bảng chính `application_train.csv` - chứa thông tin hồ sơ vay
và biến mục tiêu TARGET. Các bảng phụ xử lý riêng ở Phần Feature Engineering bảng phụ

In [ ]:
from pathlib import Path

# Resolve path theo project
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'data' / 'raw' / 'application_train.csv').exists():
    if (PROJECT_DIR.parent / 'data' / 'raw' / 'application_train.csv').exists():
        PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = str(PROJECT_DIR / 'data' / 'raw') + '/'
df = pd.read_csv(DATA_PATH + 'application_train.csv')
print(f"DATA_PATH: {DATA_PATH}")


In [ ]:
pd.set_option('display.max_columns', None)  # Hiện all
df.head(20)

Kích thước dữ liệu


In [ ]:
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

In [ ]:
df.describe()

Nhìn `describe()` có vài điểm đáng chú ý:
- `DAYS_EMPLOYED`: max=365243 (anomaly).
- `AMT_INCOME_TOTAL`: outlier cực đại (max=117M).
- `CNT_CHILDREN`: max lên tới 19.

Sẽ xử lý ở Phần 2.


## 1.2. Phân tích chi tiết các biến


### 1.2.1. Biến mục tiêu
- `TARGET = 1` (BAD): Khách hàng quá hạn ≥ 30 ngày ít nhất 1 lần trong những kỳ đầu
- `TARGET = 0` (GOOD): Các trường hợp còn lại


In [ ]:
# TARGET = 1: vỡ nợ (BAD), TARGET = 0: bình thường (GOOD)
target_counts = df['TARGET'].value_counts()
bad_rate = df['TARGET'].mean()
print(f"GOOD (0): {target_counts[0]:,} ({target_counts[0]/len(df)*100:.1f}%)")
print(f"BAD  (1): {target_counts[1]:,} ({target_counts[1]/len(df)*100:.1f}%)")
print(f"\nBad Rate: {bad_rate:.4f} ({bad_rate*100:.2f}%)")

In [ ]:
# Visulize biến mục tiêu
import matplotlib.pyplot as plt

target_counts = df['TARGET'].value_counts()
plt.figure(figsize=(6, 4))
target_counts.plot(kind='bar', color=['#2196F3', '#F44336'], edgecolor='white', linewidth=1.5)
plt.title('Phân phối Biến Mục Tiêu (TARGET)', fontsize=14, fontweight='bold')
plt.xticks([0, 1], ['GOOD (0)', 'BAD (1)'], rotation=0)
plt.ylabel('Số lượng')
for idx, v in enumerate(target_counts.values):
    plt.text(idx, v + len(df)*0.01, f'{v:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

print("\nTỷ lệ TARGET:")
print((df['TARGET'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

**Nhận xét:** Bad rate ~8% -> dataset mất cân bằng (imbalanced)


### 1.2.2. Phân tích missing value 


In [ ]:
# % missing mỗi biến
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

df_missing = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f"Tổng biến có missing: {len(df_missing)} / {df.shape[1]} biến")
print(f"Biến missing > 50%: {(df_missing['Missing %'] > 50).sum()}")
print(f"Biến missing > 70%: {(df_missing['Missing %'] > 70).sum()}")
print()
print(df_missing.head(30))


In [ ]:
# Visualize Top 30 biến có missing cao nhất
top_missing = df_missing.head(30)

fig, ax = plt.subplots(figsize=(14, 10))
# Lấy màu dựa trên giá trị Missing % của tập top_missing
colors = ['#F44336' if x > 70 else '#FF9800' if x > 50 else '#FFC107' if x > 20 else '#8BC34A'
          for x in top_missing['Missing %']]

# Xếp thứ tự biến theo Missing % giảm dần
ax.barh(top_missing.index[::-1], top_missing['Missing %'][::-1], color=colors[::-1])

# Tạo line cho các mốc quan trọng
ax.axvline(x=50, color='red', linestyle='--', alpha=0.8, label='50%')
ax.axvline(x=70, color='darkred', linestyle='--', alpha=0.8, label='70%')

ax.set_xlabel('Missing (%)')
ax.set_title(f'Missing Value Analysis — Top {len(top_missing)} biến có tỷ lệ missing cao nhất', fontsize=13, fontweight='bold')
ax.legend()

# Thêm nhãn %
for i, (idx, val) in enumerate(zip(top_missing.index[::-1], top_missing['Missing %'][::-1])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()


**Nhận xét:**
- Có 41 biến có tỷ lệ missing > 50%, đa số là các biến về nhà ở, phòng ốc 
- `EXT_SOURCE_1` missing 56% nhưng là Điểm tín dụng từ nguồn ngoài - mang signal mạnh cho dự đoán mô hình


### 1.2.3. Phân phối các biến numeric theo Good/Bad
- Xem sự khác biệt giữa GOOD và BAD trong phân phối của từng để đánh giá khả năng phân loại của biến
- Hai phân phối khác biệt/ đỉnh phân phối khác nhau -> biến có giá trị phân biệt GOOD/BAD và ngược lại


In [ ]:
# Các biến numeric quan trọng cần visualize
key_numeric = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_LAST_PHONE_CHANGE', 'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'REGION_RATING_CLIENT', 'REGION_POPULATION_RELATIVE',
    'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
    'AMT_REQ_CREDIT_BUREAU_YEAR', 'AMT_REQ_CREDIT_BUREAU_MON'
]

n_cols = 4
n_rows = (len(key_numeric) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

good = df[df['TARGET'] == 0]
bad  = df[df['TARGET'] == 1]

for i, col in enumerate(key_numeric):
    ax = axes[i]
    # Bỏ NaN khi plot
    g_vals = good[col].dropna()
    b_vals = bad[col].dropna()

    ax.hist(g_vals, bins=40, alpha=0.6, color='#2196F3', label='GOOD (0)',
            density=True, edgecolor='none')
    ax.hist(b_vals, bins=40, alpha=0.6, color='#F44336', label='BAD (1)',
            density=True, edgecolor='none')

    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)

    # Thêm missing %
    miss = df[col].isnull().mean() * 100
    if miss > 0:
        ax.set_xlabel(f'Missing: {miss:.1f}%', fontsize=8, color='red')

# Tắt các axes thừa
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Phân phối Numeric Variables theo GOOD/BAD',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Nhận xét phân phối Numeric:**
- `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3` phân tách Good/Bad rất rõ.
- `DAYS_BIRTH`, `DAYS_EMPLOYED` cho thấy người trẻ/mới đi làm rủi ro cao hơn.


### 1.2.4. Biến categorical— Bad Rate theo Từng Nhóm
- Baseline: tỷ lệ bad rate 8%
- Đánh gía biến phân loại xem trong mỗi biến, nhóm bad rate khác biệt (spread) đối với baseline không


In [ ]:
cat_cols = df.select_dtypes('object').columns.tolist()

n_cols = 2
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]

    # Tính bad rate theo từng category
    summary = df.groupby(col)['TARGET'].agg(['mean', 'count']).reset_index()
    summary.columns = [col, 'bad_rate', 'count']
    summary = summary.sort_values('bad_rate', ascending=False)

    # Bar bad rate
    bars = ax.bar(range(len(summary)), summary['bad_rate'] * 100,
                  color='#5C6BC0', edgecolor='white', linewidth=0.8)
    ax.axhline(y=df['TARGET'].mean() * 100, color='red', linestyle='--',
               alpha=0.8, label=f'Overall bad rate: {df["TARGET"].mean()*100:.1f}%')

    ax.set_xticks(range(len(summary)))
    ax.set_xticklabels(summary[col], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Bad Rate (%)')
    ax.set_title(f'{col}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

    # Label count trên mỗi bar
    for j, (bar, cnt) in enumerate(zip(bars, summary['count'])):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f'n={cnt:,}', ha='center', va='bottom', fontsize=6, rotation=90)

# Tắt axes thừa
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Bad Rate theo Categorical Variables\n(đường đỏ = overall bad rate)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Nhận xét Categorical:**
- Nam rủi ro cao hơn Nữ (10.1% vs 7.0%).
- Học vấn thấp rủi ro cao hơn.
- Thất nghiệp / Nghỉ thai sản rủi ro rất cao.


### 1.2.5. Phân tích các biến External Credit Scores (thông tin điểm tín dụng từ bên ngoài)
- Các thông tin thể thiện điểm tín dụng của khách hàng đến từ bên ngoài, có thể là CIC score, bureau score, hoặc alternative data score.
- Dùng KDE plot để xem trực quan 2 đường phân phối GOOD/BAD. Phân phối càng tách biệt -> biến càng phân biệt tốt GOOD/BAD.
- Dùng Boxplot để xem các thông tin median, IQR. median và IQR của GOOD vs BAD càng khác nhau -> biến càng phân biệt tốt GOOD/BAD.
- Tính hệ số tương quan của biến với biến target


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
good = df[df['TARGET'] == 0]
bad  = df[df['TARGET'] == 1]

for i, col in enumerate(ext_cols):
    # Row 1: KDE
    ax = axes[0, i]
    good[col].dropna().plot(kind='kde', ax=ax, color='#2196F3', label='GOOD (0)', linewidth=2)
    bad[col].dropna().plot(kind='kde', ax=ax, color='#F44336', label='BAD (1)', linewidth=2)
    ax.set_title(f'{col} — KDE', fontsize=11, fontweight='bold')
    ax.legend()
    ax.set_xlabel(col)

    # Row 2: Boxplot
    ax2 = axes[1, i]
    df.boxplot(column=col, by='TARGET', ax=ax2,
               boxprops=dict(color='#5C6BC0'),
               medianprops=dict(color='red', linewidth=2))
    ax2.set_title(f'{col} — Boxplot', fontsize=11, fontweight='bold')
    ax2.set_xlabel('TARGET (0=GOOD, 1=BAD)')
    ax2.set_ylabel(col)

plt.suptitle('EXT_SOURCE_1/2/3 — Phân phối theo GOOD/BAD',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# corr với TARGET
print("Correlation với TARGET:")
for col in ext_cols:
    corr = df[col].corr(df['TARGET'])
    print(f"  {col}: {corr:.4f}")


**Nhận xét EXT_SOURCE:**
Xếp hạng mức độ phân loại của các biến
- **EXT_SOURCE_2**: Tách biệt rõ nhất, ít missing (~0.3%)
- **EXT_SOURCE_3**: Tách biệt tốt, missing ~19%
- **EXT_SOURCE_1**: Tách biệt khá, nhưng missing ~56% 

**Nhận xét Boxplot TARGET**

Ba biến `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3` đều cho thấy nhóm GOOD (`TARGET = 0`) có median cao hơn nhóm BAD (`TARGET = 1`) -> điểm external score càng cao thì rủi ro vỡ nợ càng thấp.

- `EXT_SOURCE_3` có khoảng cách median giữa GOOD và BAD khá rõ, IQR hai nhóm cũng lệch nhau đáng kể -> biến có khả năng phân biệt GOOD/BAD tốt.
- `EXT_SOURCE_1` cũng cho thấy median GOOD cao hơn BAD và hai box có sự dịch chuyển rõ -> biến có tín hiệu phân biệt tốt.
- `EXT_SOURCE_2` có median khác nhau nhưng IQR của hai nhóm overlap nhiều hơn -> vẫn có tín hiệu, nhưng mức phân tách nhìn từ boxplot yếu hơn hai biến còn lại.


**Hệ số tương quan Correlation của 3 biến đều âm với TARGET** (thể hiện giá trị EXT_SOURCE càng cao thì ít rủi ro hơn)

### 1.2.6. Phân tích các biến thời gian 
 - DAYS_BIRTH: Tuổi
 - DAYS_EMPLOYED: Thâm niên công việc

Các biến `DAYS_*` trong dataset đo số ngày **trước ngày nộp đơn** -> giá trị âm.
-> convert sang đơn vị năm bằng hàm `abs(DAYS_X) / 365`.

**Biến `DAYS_EMPLOYED`:** có giá trị bất thường `365243` (~1000 năm) là **mã hóa đặc biệt** cho người không có việc làm chính thức (thất nghiệp, nội trợ, hưu trí, tự kinh doanh...). Không xử lý -> outlier cực đoan. -> replace 365243 bằng NaN, xử lý missing riêng.


In [ ]:
# Chỉ dùng biến tạm để vẽ EDA, không lưu đè df gốc (tránh sinh lỗi Feature Engineering phía sau)
_age_years = abs(df['DAYS_BIRTH']) / 365
_employed_years = abs(df['DAYS_EMPLOYED'].replace(365243, float('nan'))) / 365

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. Plot Phân bố độ tuổi theo TARGET
sns.kdeplot(_age_years[df['TARGET'] == 0], label='TARGET = 0 (Good)', color='#2196F3', fill=True, alpha=0.3, ax=ax1)
sns.kdeplot(_age_years[df['TARGET'] == 1], label='TARGET = 1 (Bad)', color='#F44336', fill=True, alpha=0.3, ax=ax1)
ax1.set_title('Phân bố Độ tuổi Khách hàng (Age in Years)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Tuổi (Năm)')
ax1.set_ylabel('Density')
ax1.legend()

# 2. Plot Phân bố thâm niên làm việc theo TARGET
sns.kdeplot(_employed_years[df['TARGET'] == 0], label='TARGET = 0 (Good)', color='#2196F3', fill=True, alpha=0.3, ax=ax2)
sns.kdeplot(_employed_years[df['TARGET'] == 1], label='TARGET = 1 (Bad)', color='#F44336', fill=True, alpha=0.3, ax=ax2)
ax2.set_title('Phân bố Thâm niên Làm việc (Employed Years)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Số năm làm việc (đã xử lý outlier)')
ax2.set_ylabel('Density')
ax2.legend()

plt.tight_layout()
plt.show()

**Nhận xét thời gian:**
- Tuổi trẻ (20-30) có tỷ lệ Bad cao nhất (~11%).
- Mới đi làm (<2 năm) rủi ro cao (~10%).


In [ ]:
print("THỐNG KÊ TRUNG BÌNH THEO NHÓM KHÁCH HÀNG:")
print(f"Tuổi trung bình (Good - 0): {_age_years[df['TARGET'] == 0].mean():.1f} tuổi")
print(f" Tuổi trung bình (Bad  - 1): {_age_years[df['TARGET'] == 1].mean():.1f} tuổi")
print(f"Thâm niên làm việc trung bình (Good - 0): {_employed_years[df['TARGET'] == 0].mean():.1f} năm")
print(f"Thâm niên làm việc trung bình (Bad  - 1): {_employed_years[df['TARGET'] == 1].mean():.1f} năm")

### 1.2.7. Phân tích các biến tài chính


In [ ]:
amt_cols = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(amt_cols):
    ax = axes[i]
    good_vals = df[df['TARGET'] == 0][col].dropna()
    bad_vals  = df[df['TARGET'] == 1][col].dropna()

    ax.hist(good_vals, bins=80, alpha=0.6, color='#2196F3', label='GOOD', density=True)
    ax.hist(bad_vals,  bins=80, alpha=0.6, color='#F44336', label='BAD',  density=True)
    ax.axvline(good_vals.mean(), color='#1565C0', linestyle='--', linewidth=1.5,
               label=f'GOOD mean={good_vals.mean()/1e6:.2f}M')
    ax.axvline(bad_vals.mean(),  color='#B71C1C', linestyle='--', linewidth=1.5,
               label=f'BAD  mean={bad_vals.mean()/1e6:.2f}M')
    ax.set_title(f'{col} — RAW', fontsize=11, fontweight='bold')
    ax.set_xlabel(f'{col} (đơn vị gốc)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Phân phối AMT Variables theo GOOD/BAD',fontsize=12, fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()

Các biến `AMT_*` đo giá trị tiền tệ — thường có phân phối **right-skewed** (lệch phải) vì:
- Đa số khách hàng vay số tiền nhỏ → phân phối tập trung trái
- Một số ít vay số tiền rất lớn → đuôi phải dài


Log transform để xem  phân phối cân xứng hơn


In [ ]:
amt_cols = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(amt_cols):
    ax = axes[i]
    # Dùng log scale vì phân phối right-skewed
    good_vals = np.log1p(df[df['TARGET'] == 0][col].dropna())
    bad_vals  = np.log1p(df[df['TARGET'] == 1][col].dropna())

    ax.hist(good_vals, bins=50, alpha=0.6, color='#2196F3', label='GOOD', density=True)
    ax.hist(bad_vals,  bins=50, alpha=0.6, color='#F44336', label='BAD',  density=True)
    ax.set_title(f'{col} (log scale)', fontsize=11, fontweight='bold')
    ax.set_xlabel(f'log({col})')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Phân phối AMT Variables (log scale) theo GOOD/BAD',fontsize=12, fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()


**Nhận xét:** Các biến `AMT_*` right-skewed nặng. Log transform giúp phân phối chuẩn hơn.


## 1.3. Tổng kết EDA


In [ ]:
# Tổng hợp: Top biến correlate với TARGET
num_cols = df.select_dtypes(['int64', 'float64']).columns.tolist()
corr_with_target = df[num_cols].corr()['TARGET'].drop('TARGET')
top20_idx = corr_with_target.abs().sort_values(ascending=False).head(20).index

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart thể hiện correlation với TARGET
colors = ['#F44336' if v < 0 else '#2196F3' for v in corr_with_target[top20_idx].values]
axes[0].barh(top20_idx[::-1], corr_with_target[top20_idx][::-1], color=colors[::-1])
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Pearson Correlation với TARGET')
axes[0].set_title('Top 20 Biến — Correlation với TARGET', fontweight='bold')

# Correlation matrix với top 10 biến
top10 = corr_with_target.abs().sort_values(ascending=False).head(10).index
corr_matrix = df[top10].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=axes[1],
            annot_kws={'size': 9}, linewidths=0.5)
axes[1].set_title('Correlation Matrix — Top 10 Features', fontweight='bold')

plt.tight_layout()
plt.show()

### Tổng kết sau EDA:
- **Bad rate ~8.07%**: Dữ liệu imbalanced, cần dùng GINI/AUC/KS; LR Scorecard cuối không dùng `class_weight` để xác suất còn diễn giải được như PD.
- **Signal**: EXT_SOURCE và độ tuổi/thâm niên mạnh nhất.
- **Cleaning**: Outliers cần xử lý.


# 2. Tiền xử lý dữ liệu và Feature Engineering (bảng dataset chính)


## 2.1 — Xử lý giá trị bất thường (anormal)
- `DAYS_EMPLOYED = 365243`: Giá trị mã hóa cho người không đi làm, không phải dữ liệu thực
- `CODE_GENDER = 'XNA'`: giá trị không hợp lệ, chỉ có 4 records


In [ ]:
# Clean Anomalies
print("TRƯỚC KHI CLEAN")
print(f"DAYS_EMPLOYED = 365243: {(df['DAYS_EMPLOYED'] == 365243).sum():,} records")
print(f"CODE_GENDER = XNA: {(df['CODE_GENDER'] == 'XNA').sum()} records")

# Fix DAYS_EMPLOYED anomaly
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# Fix CODE_GENDER XNA
df['CODE_GENDER'] =df['CODE_GENDER'].replace("XNA", np.nan)

print("\nSAU KHI CLEAN")
print(f"DAYS_EMPLOYED NaN: {df['DAYS_EMPLOYED'].isnull().sum():,}")
print(f"CODE_GENDER NaN: {df['CODE_GENDER'].isnull().sum()}")

**Nhận xét:**
Sau bước này, `DAYS_EMPLOYED` sẽ có thêm ~55,000 NaN mới — đây là giá trị NaN có ý nghĩa về mặt nghiệp vụ
(người không đi làm) -> Sẽ xử lý imputation sau


## 2.2. Convert Biến Thời Gian sang Đơn Vị Năm
`DAYS_*` là số âm (ngày tính ngược). Convert sang năm để:
- Dễ diễn giải (tuổi 35 thay vì -12775)
- Giúp các ratio feature có đơn vị tự nhiên hơn


In [ ]:
# Convert DAYS sang năm
days_cols = ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
             'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']

for col in days_cols:
    new_col = col.replace('DAYS_', '') + '_YEARS'
    df[new_col] = abs(df[col]) / 365

print("Các biến mới đã tạo (đơn vị năm):")
year_cols = [c.replace('DAYS_', '') + '_YEARS' for c in days_cols]
print(df[year_cols].describe().round(2))

**Nhận xét:**
- `BIRTH_YEARS`: tuổi khách hàng -> range hợp lý 20-69 tuổi
- `EMPLOYED_YEARS` -> sau khi clean anomaly, range từ 0-49 năm
- Các biến gốc `DAYS_*` vẫn giữ để dùng cho feature engineering sau


## 2.3. Tạo Flag "IS_MISSING" cho các biến missing có ý nghĩa
**Lý do*:Các biến bị missing không phải ngẫu nhiên (Missing Not at Random - MNAR) 
Ví dụ:
- `EXT_SOURCE_1` missing -> Do khách hàng không có lịch sử tín dụng tại nguồn đó -> rủi ro hơn
- `DAYS_EMPLOYED` = 365243 (đã replace NaN) → không có việc làm ổn định -> rủi ro hơn
- `OCCUPATION_TYPE` missing → không khai báo nghề → có thể là dấu hiệu che giấu nghề nghiệp

=> Tạo biến binary `X_IS_MISSING` để model học được pattern này.


In [ ]:
# Các biến có missing có ý nghĩa
meaningful_missing = [
    'EXT_SOURCE_1',        # credit score nguồn 1
    'EXT_SOURCE_2',        # credit score nguồn 2
    'EXT_SOURCE_3',        # credit score nguồn 3
    'OCCUPATION_TYPE',     # nghề nghiệp 
    'AMT_GOODS_PRICE',     # giá hàng hóa
    'OWN_CAR_AGE',         # tuổi xe — missing khi không có xe (FLAG_OWN_CAR = N)
]

# Tạo flag TRƯỚC khi impute (impute sẽ xóa NaN, không tạo flag được)
for col in meaningful_missing:
    flag_col = col + '_IS_MISSING'
    df[flag_col] = df[col].isnull().astype(int)
    n_missing = df[flag_col].sum()
    print(f"  {flag_col}: {n_missing:,} ({n_missing/len(df)*100:.1f}%)")

# Flag cho DAYS_EMPLOYED anomaly
df['DAYS_EMPLOYED_ANOMALY'] = df['DAYS_EMPLOYED'].isnull().astype(int)
print(f"\n  DAYS_EMPLOYED_ANOMALY: {df['DAYS_EMPLOYED_ANOMALY'].sum():,} ({df['DAYS_EMPLOYED_ANOMALY'].mean()*100:.1f}%)")

**Nhận xét**:

Các flag này sẽ được đưa vào model như những feature độc lập. LightGBM có thể
tự học được "khi EXT_SOURCE_1 missing thì rủi ro cao hơn bao nhiêu".

**Note**:

Với WOE/IV ở Phần 4: missing sẽ được xử lý như một bin riêng — nên thực ra
không nhất thiết phải tạo flag cho model Logistic Regression.


## 2.4. Tách Train / Test Set


Giải pháp Out-Of-Time (OOT) Split thông qua biến `Proxy DAYS_DECISION`

1. Vấn đề của Random Split (Data Leakage về mặt thời gian) Trong bài toán thực tế của ngân hàng (Credit Scoring), hành vi vay nợ và vỡ nợ của khách hàng luôn thay đổi theo chu kỳ kinh tế (Data Drift). Việc sử dụng train_test_split ngẫu nhiên sẽ lấy dữ liệu tương lai để huấn luyện và đi dự đoán quá khứ, làm mất đi tính liên tục của thời gian và dẫn đến hiện tượng Temporal Data Leakage (mô hình cho GINI/AUC rất cao nhưng fail khi đưa ra thực tế).

Nguyên tắc: Huấn luyện trên dữ liệu QUÁ KHỨ và kiểm thử trên dữ liệu TƯƠNG LAI (Out-Of-Time Validation).

2. Khó khăn của bộ dữ liệu Home Credit Bảng dữ liệu chính application_train hoàn toàn không có cột Ngày tháng (Date/Timestamp) để chẻ data theo thời gian thực. Các biến như DAYS_BIRTH hay DAYS_EMPLOYED chỉ nói lên độ tuổi cá nhân chứ không nói lên thời điểm họ nộp hồ sơ.

3. Giải pháp Proxy OOT bằng DAYS_DECISION:

- Truy xuất vào bảng previous_application.csv – nơi ghi nhận các mốc thời gian khách hàng đã nộp hồ sơ trước đó thông qua cột DAYS_DECISION (số ngày tính từ quyết định vay trước đó đến thời điểm nộp hồ sơ hiện tại).
- Extract: Với mỗi khách hàng (SK_ID_CURR),lấy ra quyết định gần nhất max(DAYS_DECISION) (giá trị mang số âm nhỏ nhất, gần số 0 nhất).
- Sort: Map biến này vào bảng chính và sắp xếp (sort_values(ascending=True)) từ người có lịch sử lâu đời nhất (âm nhiều) đến người mới nộp hồ sơ gần đây nhất (gần 0).
- Split: Lấy 80% dữ liệu cũ nhất làm tập Train và 20% dữ liệu mới nhất làm tập Test (OOT).
(Với những khách hàng không có lịch sử previous_application, tạm fill bằng giá trị min - coi như họ thuộc nhóm khách hàng xa nhất, cho vào tập Train, đảm bảo tập OOT Test là những hồ sơ mới).


In [ ]:
# Train/Test Split (OOT Proxy)
# Dùng DAYS_DECISION từ previous_application để sort khách hàng theo thời gian (OOT split)
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns if c not in ['SK_ID_CURR', 'TARGET']]
X = df[feature_cols]
y = df['TARGET']

# Đọc bảng previous_application để lấy DAYS_DECISION gần nhất
prev_app_path = DATA_PATH + 'previous_application.csv'
prev_app = pd.read_csv(prev_app_path, usecols=['SK_ID_CURR', 'DAYS_DECISION'])

# Lấy quyết định gần nhất (giá trị âm lớn nhất / gần 0 nhất)
last_decision = prev_app.groupby('SK_ID_CURR')['DAYS_DECISION'].max().reset_index()

# Merge vào df chính để lấy thời gian
time_proxy_df = df[['SK_ID_CURR']].merge(last_decision, on='SK_ID_CURR', how='left')

# Fill NaN cho những KH chưa từng có previous_application bằng giá trị min (xa nhất)
min_days = time_proxy_df['DAYS_DECISION'].min()
time_proxy_df['DAYS_DECISION'] = time_proxy_df['DAYS_DECISION'].fillna(min_days)

# Sort theo thời gian: từ xa nhất trong quá khứ (số âm lớn) đến gần nhất (gần 0)
sort_idx = time_proxy_df.sort_values(by='DAYS_DECISION', ascending=True).index
X = X.reindex(sort_idx)
y = y.reindex(sort_idx)

# Lấy 80% đầu tiên làm Train (quá khứ), 20% cuối cùng làm Test (tương lai - OOT)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print(f"X_train (OOT): {X_train.shape} | X_test (OOT): {X_test.shape}")
print(f"Train bad rate: {y_train.mean()*100:.2f}% | Test/OOT bad rate: {y_test.mean()*100:.2f}%")


## 2.5. Xử Lý Missing Values (Imputation)
- Do nhiều biến có missing pattern phức tạp, xử lý trước: median cho numeric, 'Missing' cho categorical.
- tính median trên Train rồi áp cho Test -> Tránh Data Leakage.


In [ ]:
num_cols = X_train.select_dtypes(include=['float64', 'int64', 'number']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
# 1. Nhóm các biến Numeric quan trọng / Missing nhiều (Sẽ dùng -999 để tạo Bin riêng khi tính WOE)
special_numeric = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_EMPLOYED','EMPLOYED_YEARS','OWN_CAR_AGE']
# 2. Xử lý biến Numeric
for col in num_cols:
    if col in X_train.columns:
        if col in special_numeric:
            # Điền -999 để tách riêng nhóm Missing khi tính WOE
            X_train[col] = X_train[col].fillna(-999)
            X_test[col]  = X_test[col].fillna(-999)
        else:
            # Điền Median cho các biến thiếu ít
            med = X_train[col].median()
            X_train[col] = X_train[col].fillna(med)
            X_test[col]  = X_test[col].fillna(med)
# 3. Xử lý biến Categorical
for col in cat_cols:
    if col in X_train.columns:
        if X_train[col].dtype.name == 'category':
            if 'Missing' not in X_train[col].cat.categories:
                X_train[col] = X_train[col].cat.add_categories('Missing')
                X_test[col]  = X_test[col].cat.add_categories('Missing')
                
        X_train[col] = X_train[col].fillna('Missing')
        X_test[col]  = X_test[col].fillna('Missing')
print("Đã Impute xong. Các biến num quan trọng dùng -999, các biến num khác dùng median, biến cat dùng 'Missing'.")

In [ ]:
print(f"Total missing train: {X_train.isnull().sum().sum():,}")
print (f"Total missing test: {X_test.isnull().sum().sum():,}")

## 2.6. Xử lý outlier

Không xử lý outlier trước WOE/LR Scorecard.

Lý do:
- WOE binning đã gom extreme values vào các bin biên.
--> Cân nhắc xử lý outlier riêng trong thực nghiệm LightGBM nếu cần.

## 2.7. Feature engineering

In [ ]:
# Tạo features sau split
def create_features(X_data):
    X_data = X_data.copy()

    # 1. Các biến tỷ lệ tài chính
    X_data['ANNUITY_TO_INCOME'] = X_data['AMT_ANNUITY'] / X_data['AMT_INCOME_TOTAL'].replace(0, np.nan)
    X_data['CREDIT_TO_INCOME']  = X_data['AMT_CREDIT'] / X_data['AMT_INCOME_TOTAL'].replace(0, np.nan)
    X_data['CREDIT_TO_GOODS']   = X_data['AMT_CREDIT'] / X_data['AMT_GOODS_PRICE'].replace(0, np.nan)
    X_data['INCOME_PER_PERSON'] = X_data['AMT_INCOME_TOTAL'] / X_data['CNT_FAM_MEMBERS'].replace(0, np.nan)
    X_data['ANNUITY_TO_CREDIT'] = X_data['AMT_ANNUITY'] / X_data['AMT_CREDIT'].replace(0, np.nan)

    # 2. EXT_SOURCE aggregate
    # Lưu ý: EXT_SOURCE missing đã được impute = -999 để WOE bin riêng.
    # Khi tạo aggregate, đổi -999 về NaN để không làm méo mean/min/max/product.
    ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    ext_clean = X_data[ext_cols].replace(-999, np.nan)

    X_data['EXT_SOURCE_MEAN'] = ext_clean.mean(axis=1)
    X_data['EXT_SOURCE_MIN']  = ext_clean.min(axis=1)
    X_data['EXT_SOURCE_MAX']  = ext_clean.max(axis=1)
    X_data['EXT_SOURCE_STD']  = ext_clean.std(axis=1)

    # Product chỉ tính khi có đủ 3 EXT_SOURCE, thiếu thì để NaN và xử lý ở sanity check sau
    X_data['EXT_SOURCE_PRODUCT'] = ext_clean.prod(axis=1, min_count=3)

    # Missing count lấy từ flags nếu có, nếu không thì tính từ ext_clean
    missing_flag_cols = [f'{c}_IS_MISSING' for c in ext_cols if f'{c}_IS_MISSING' in X_data.columns]
    if len(missing_flag_cols) == 3:
        X_data['EXT_SOURCE_MISSING_COUNT'] = X_data[missing_flag_cols].sum(axis=1)
    else:
        X_data['EXT_SOURCE_MISSING_COUNT'] = ext_clean.isnull().sum(axis=1)

    # 3. Biến thời gian
    employed_years_clean = X_data['EMPLOYED_YEARS'].replace(-999, np.nan)
    X_data['EMPLOYED_TO_AGE'] = employed_years_clean / X_data['BIRTH_YEARS'].replace(0, np.nan)
    X_data['BIRTH_EMPLOYED_DIFF'] = X_data['BIRTH_YEARS'] - employed_years_clean

    if 'DAYS_LAST_PHONE_CHANGE' in X_data.columns:
        X_data['RECENT_PHONE_CHANGE'] = (X_data['DAYS_LAST_PHONE_CHANGE'] >= -90).astype(int)

    # 4. Document flag aggregation
    doc_cols = [c for c in X_data.columns if c.startswith('FLAG_DOCUMENT')]
    if doc_cols:
        X_data['DOCUMENT_SUM'] = X_data[doc_cols].sum(axis=1)

    return X_data


X_train = create_features(X_train)
X_test  = create_features(X_test)

# Fill NaN phát sinh từ feature engineering bằng median của train
new_num_cols = X_train.select_dtypes(include=['float64', 'int64', 'number']).columns

for col in new_num_cols:
    if X_train[col].isnull().sum() > 0:
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        X_test[col]  = X_test[col].fillna(med)
print("Đã tạo feature engineering từ bảng chính.")

In [ ]:
# Group-level Organization type x name income type / K-Fold Target Encoding
from sklearn.model_selection import KFold

# Tạo biến nhóm tạm thời
X_train['ORG_INCOME_GROUP'] = (
    X_train['ORGANIZATION_TYPE'].astype(str) + '_' +
    X_train['NAME_INCOME_TYPE'].astype(str)
)

X_test['ORG_INCOME_GROUP'] = (
    X_test['ORGANIZATION_TYPE'].astype(str) + '_' +
    X_test['NAME_INCOME_TYPE'].astype(str)
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
X_train['ORG_INCOME_BAD_RATE'] = np.nan

# Out-of-fold encoding cho train để tránh target leakage
for tr_idx, val_idx in kf.split(X_train):
    X_tr_fold = X_train.iloc[tr_idx]
    y_tr_fold = y_train.iloc[tr_idx]

    mean_target = y_tr_fold.groupby(X_tr_fold['ORG_INCOME_GROUP']).mean()

    X_train.iloc[
        val_idx,
        X_train.columns.get_loc('ORG_INCOME_BAD_RATE')
    ] = X_train.iloc[val_idx]['ORG_INCOME_GROUP'].map(mean_target)

# Fill nhóm hiếm/unseen bằng global bad rate
global_bad_rate = y_train.mean()
X_train['ORG_INCOME_BAD_RATE'] = X_train['ORG_INCOME_BAD_RATE'].fillna(global_bad_rate)

# Encode test bằng mapping từ toàn bộ train
train_mean_target = y_train.groupby(X_train['ORG_INCOME_GROUP']).mean()
X_test['ORG_INCOME_BAD_RATE'] = (
    X_test['ORG_INCOME_GROUP']
    .map(train_mean_target)
    .fillna(global_bad_rate)
)

# Drop group raw để tránh high-cardinality WOE / overfit
X_train = X_train.drop(columns=['ORG_INCOME_GROUP'], errors='ignore')
X_test  = X_test.drop(columns=['ORG_INCOME_GROUP'], errors='ignore')

# Drop DAYS_ID_PUBLISH khỏi features
X_train = X_train.drop(columns=['DAYS_ID_PUBLISH'], errors='ignore')
X_test  = X_test.drop(columns=['DAYS_ID_PUBLISH'], errors='ignore')

print("Đã tạo ORG_INCOME_BAD_RATE bằng K-Fold Target Encoding.")
print("Đã drop ORG_INCOME_GROUP và DAYS_ID_PUBLISH khỏi features.")


## 2.8 Kiểm Tra Lại Sau Preprocessing


In [ ]:
# SANITY CHECK & FINAL CLEANING

# 1. Quét dọn toàn bộ Inf (thường sinh ra do phép chia) thành NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# 2. Xử lý triệt để NaN (sinh ra từ Feature Engineering)
nan_check_train = X_train.isnull().sum().sum()
if nan_check_train > 0:
    num_cols_only = X_train.select_dtypes(include=['float64', 'int64', 'number']).columns
    # LUÔN tính median từ X_train
    medians = X_train[num_cols_only].median()
    
    # Fill cho cả Train và Test
    X_train[num_cols_only] = X_train[num_cols_only].fillna(medians)
    X_test[num_cols_only]  = X_test[num_cols_only].fillna(medians)


# 3. In báo cáo tổng kết cuối cùng
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Số lượng features thực tế đưa vào model: {X_train.shape[1]}")
print("-" * 40)
print(f"Missing còn sót lại (Train): {X_train.isnull().sum().sum()}  {'[TỐT]' if X_train.isnull().sum().sum() == 0 else '[LỖI]'}")
print(f"Missing còn sót lại (Test) : {X_test.isnull().sum().sum()}  {'[TỐT]' if X_test.isnull().sum().sum() == 0 else '[LỖI]'}")
print(f"Inf còn sót lại (Train)    : {np.isinf(X_train.select_dtypes('number')).sum().sum()}  {'[TỐT]' if np.isinf(X_train.select_dtypes('number')).sum().sum() == 0 else '[LỖI]'}")

# 3. Feature Engineering từ Bảng Phụ
6 bảng phụ chứa **lịch sử hành vi tín dụng** của khách hàng

**Chiến lược**: Aggregate (tổng hợp) mỗi bảng phụ về cấp độ `SK_ID_CURR`,
rồi JOIN vào bảng chính. Mỗi bảng phụ sẽ tạo ra ~10-30 features mới.


## 3.1. Load các bảng phụ


In [ ]:
# Load bảng phụ
import gc  # garbage collector — giải phóng RAM sau khi dùng xong

DATA_PATH = str(PROJECT_DIR / 'data' / 'raw') + '/'

print("Loading auxiliary tables...")
bureau          = pd.read_csv(DATA_PATH + 'bureau.csv')
bureau_balance  = pd.read_csv(DATA_PATH + 'bureau_balance.csv')
prev_app        = pd.read_csv(DATA_PATH + 'previous_application.csv')
pos_cash        = pd.read_csv(DATA_PATH + 'POS_CASH_balance.csv')
installments    = pd.read_csv(DATA_PATH + 'installments_payments.csv')
credit_card     = pd.read_csv(DATA_PATH + 'credit_card_balance.csv')

print(f"bureau:         {bureau.shape}")
print(f"bureau_balance: {bureau_balance.shape}")
print(f"prev_app:       {prev_app.shape}")
print(f"pos_cash:       {pos_cash.shape}")
print(f"installments:   {installments.shape}")
print(f"credit_card:    {credit_card.shape}")

## 3.2. Bureau.csv + Bureau_Balance.csv — Lịch sử CIC
**Ý nghĩa nghiệp vụ:**
`bureau.csv` chứa tất cả khoản tín dụng của khách hàng tại các tổ chức khác
(được báo cáo lên CIC).

`bureau_balance.csv` chứa trạng thái hàng tháng của mỗi khoản tín dụng đó,
trong đó cột `STATUS`:
- `C` = Closed (đóng)
- `X` = Unknown
- `0` = Không quá hạn
- `1-5` = Quá hạn 1-30, 31-60, 61-90, 91-120, 120+ ngày

**aggregate:**
- `bureau_balance` → aggregate theo `SK_ID_BUREAU` → merge vào `bureau`
- `bureau` (đã gộp balance) → aggregate theo `SK_ID_CURR` → join vào df chính


In [ ]:
# Aggregate bureau_balance -> bureau level

# Đếm số tháng theo từng STATUS (hành vi trả nợ theo thời gian)
bb_status = pd.get_dummies(bureau_balance[['SK_ID_BUREAU', 'STATUS']], 
                            columns=['STATUS'])
bb_agg = bb_status.groupby('SK_ID_BUREAU').agg('sum').reset_index()
bb_agg.columns = ['SK_ID_BUREAU'] + ['BB_STATUS_' + c.split('_')[1] 
                                      for c in bb_agg.columns[1:]]

# Tổng số tháng trong lịch sử
bb_months = bureau_balance.groupby('SK_ID_BUREAU')['MONTHS_BALANCE'].agg(
    BB_MONTHS_COUNT='count',
    BB_MONTHS_MIN='min'
).reset_index()

bureau_balance_agg = bb_months.merge(bb_agg, on='SK_ID_BUREAU', how='left')

print(f"bureau_balance aggregated: {bureau_balance_agg.shape}")
# Giải phóng RAM
del bureau_balance, bb_status, bb_agg, bb_months
gc.collect()

In [ ]:
# Merge balance vào bureau, sau đó aggregate -> SK_ID_CURR level

bureau_full = bureau.merge(bureau_balance_agg, on='SK_ID_BUREAU', how='left')

# Aggregate theo SK_ID_CURR
bureau_agg = bureau_full.groupby('SK_ID_CURR').agg(
    # Số lượng khoản vay 
    BUREAU_LOAN_COUNT            = ('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT          = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_CLOSED_COUNT          = ('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),

    # Thông tin quá hạn
    BUREAU_OVERDUE_MAX           = ('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_OVERDUE_MEAN          = ('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_AMT_OVERDUE_MAX       = ('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_AMT_OVERDUE_MEAN      = ('AMT_CREDIT_MAX_OVERDUE', 'mean'),

    # Số tiền tín dụng
    BUREAU_AMT_CREDIT_SUM        = ('AMT_CREDIT_SUM', 'sum'),
    BUREAU_AMT_CREDIT_MEAN       = ('AMT_CREDIT_SUM', 'mean'),
    BUREAU_AMT_DEBT_SUM          = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_AMT_DEBT_MEAN         = ('AMT_CREDIT_SUM_DEBT', 'mean'),

    # Thời gian
    BUREAU_DAYS_CREDIT_MAX       = ('DAYS_CREDIT', 'max'),
    BUREAU_DAYS_CREDIT_MEAN      = ('DAYS_CREDIT', 'mean'),
    BUREAU_DAYS_ENDDATE_MAX      = ('DAYS_CREDIT_ENDDATE', 'max'),
    BUREAU_CNT_PROLONG_SUM       = ('CNT_CREDIT_PROLONG', 'sum'),
).reset_index()

# Feature phái sinh từ bureau
bureau_agg['BUREAU_ACTIVE_RATIO'] = (
    bureau_agg['BUREAU_ACTIVE_COUNT'] / bureau_agg['BUREAU_LOAN_COUNT'].clip(lower=1)
)
bureau_agg['BUREAU_DEBT_RATIO'] = (
    bureau_agg['BUREAU_AMT_DEBT_SUM'] / bureau_agg['BUREAU_AMT_CREDIT_SUM'].clip(lower=1)
)

print(f"bureau_agg shape: {bureau_agg.shape}")
print(f"Features tạo từ bureau: {bureau_agg.shape[1] - 1}")

del bureau_full, bureau_balance_agg, bureau
gc.collect()

**Giải thích các features**

| Feature | Ý nghĩa kinh doanh |
|---------|-------------------|
| `BUREAU_ACTIVE_COUNT` | Đang có bao nhiêu khoản vay cùng lúc -> gánh nặng tín dụng |
| `BUREAU_OVERDUE_MAX` | Số ngày quá hạn nhiều nhất từ trước đến nay -> red flag |
| `BUREAU_AMT_OVERDUE_MAX` | Số tiền quá hạn tối đa -> mức độ nghiêm trọng |
| `BUREAU_DEBT_RATIO` | Tỷ lệ dư nợ / hạn mức -> utilization rate |
| `BUREAU_CNT_PROLONG_SUM` | Số lần gia hạn -> dấu hiệu khó khăn tài chính |


## 3.3. Previous_Application.csv
 Lịch Sử Vay tại Home Credit, thể hiện:
- Lịch sử approved/refused/canceled
- Số lần apply và tỷ lệ được duyệt


In [ ]:
# Aggregate previous_application

prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    # Số lần apply
    PREV_APP_COUNT               = ('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT          = ('NAME_CONTRACT_STATUS',
                                    lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT           = ('NAME_CONTRACT_STATUS',
                                    lambda x: (x == 'Refused').sum()),
    PREV_CANCELED_COUNT          = ('NAME_CONTRACT_STATUS',
                                    lambda x: (x == 'Canceled').sum()),

    # Thông tin số tiền
    PREV_AMT_APPLICATION_MEAN    = ('AMT_APPLICATION', 'mean'),
    PREV_AMT_APPLICATION_MAX     = ('AMT_APPLICATION', 'max'),
    PREV_AMT_CREDIT_MEAN         = ('AMT_CREDIT', 'mean'),
    PREV_AMT_CREDIT_MAX          = ('AMT_CREDIT', 'max'),
    PREV_AMT_DOWN_PAYMENT_MEAN   = ('AMT_DOWN_PAYMENT', 'mean'),

    # Thời gian
    PREV_DAYS_DECISION_MAX       = ('DAYS_DECISION', 'max'),  # lần gần nhất
    PREV_DAYS_DECISION_MEAN      = ('DAYS_DECISION', 'mean'),

    # Rate
    PREV_RATE_DOWN_PAYMENT_MEAN  = ('RATE_DOWN_PAYMENT', 'mean'),
    PREV_RATE_INTEREST_MEAN      = ('RATE_INTEREST_PRIMARY', 'mean'),
).reset_index()

# Feature phái sinh
prev_agg['PREV_APPROVAL_RATE'] = (
    prev_agg['PREV_APPROVED_COUNT'] / prev_agg['PREV_APP_COUNT'].clip(lower=1)
)
prev_agg['PREV_CREDIT_TO_APP_RATIO'] = (
    prev_agg['PREV_AMT_CREDIT_MEAN'] / prev_agg['PREV_AMT_APPLICATION_MEAN'].clip(lower=1)
)

print(f"prev_agg shape: {prev_agg.shape}")
print(f"Features tạo từ previous_application: {prev_agg.shape[1] - 1}")

del prev_app
gc.collect()

** Các Feature quan trọng:**
- `PREV_APPROVAL_RATE` : số lần được duyệt / tổng số lần apply
  -> Tỷ lệ thấp : bị từ chối nhiều lần
- `PREV_REFUSED_COUNT` : số lần bị từ chối hoàn toàn
  -> Cao: lịch sử tín dụng xấu


## 3.4. POS_CASH_balance.csv — Hành Vi Trả Nợ POS/Tiền Mặt
Chứa trạng thái hàng tháng của các khoản vay POS và vay tiền mặt trước đây, trong đó có `SK_DPD`  (Days Past Due — số ngày quá hạn) -> phản ánh về hành vi trả nợ


In [ ]:
# Aggregate POS_CASH_balance

pos_agg = pos_cash.groupby('SK_ID_CURR').agg(
    POS_MONTHS_COUNT             = ('MONTHS_BALANCE', 'count'),
    POS_SK_DPD_MAX               = ('SK_DPD', 'max'),
    POS_SK_DPD_MEAN              = ('SK_DPD', 'mean'),
    POS_SK_DPD_DEF_MAX           = ('SK_DPD_DEF', 'max'),
    POS_SK_DPD_DEF_MEAN          = ('SK_DPD_DEF', 'mean'),
    POS_COMPLETED_COUNT          = ('NAME_CONTRACT_STATUS',
                                    lambda x: (x == 'Completed').sum()),
    POS_ACTIVE_COUNT             = ('NAME_CONTRACT_STATUS',
                                    lambda x: (x == 'Active').sum()),
).reset_index()

# Tỷ lệ tháng có DPD > 0
pos_dpd_months = pos_cash[pos_cash['SK_DPD'] > 0].groupby('SK_ID_CURR').size().reset_index()
pos_dpd_months.columns = ['SK_ID_CURR', 'POS_DPD_MONTHS_COUNT']
pos_agg = pos_agg.merge(pos_dpd_months, on='SK_ID_CURR', how='left')
pos_agg['POS_DPD_MONTHS_COUNT'].fillna(0, inplace=True)
pos_agg['POS_DPD_RATE'] = pos_agg['POS_DPD_MONTHS_COUNT'] / pos_agg['POS_MONTHS_COUNT'].clip(lower=1)

print(f"pos_agg shape: {pos_agg.shape}")
print(f"Features tạo từ pos: {pos_agg.shape[1] - 1}")
del pos_cash, pos_dpd_months
gc.collect()

## 3.5. Installments_Payments.csv — Lịch Sử Trả Góp
**chi tiết nhất về hành vi trả nợ**: thể hiện ngày phải trả,
ngày thực tế trả, số tiền phải trả và số tiền thực tế trả.

Từ đó có thể tính toán:
- Trả muộn bao nhiêu ngày: `DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT`
- Trả thiếu bao nhiêu: `AMT_INSTALMENT - AMT_PAYMENT`
- Tỷ lệ trả đúng hạn


In [ ]:
# Aggregate installments_payments

# Tính days late (dương: trả muộn, âm: trả sớm)
installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
installments['AMT_SHORTAGE'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']
installments['PAYMENT_RATIO'] = installments['AMT_PAYMENT'] / installments['AMT_INSTALMENT'].clip(lower=1)

inst_agg = installments.groupby('SK_ID_CURR').agg(
    INST_COUNT                   = ('NUM_INSTALMENT_NUMBER', 'count'),

    # Trễ hạn
    INST_DAYS_LATE_MAX           = ('DAYS_LATE', 'max'),
    INST_DAYS_LATE_MEAN          = ('DAYS_LATE', 'mean'),
    INST_DAYS_LATE_SUM           = ('DAYS_LATE', 'sum'),

    # Thiếu tiền
    INST_AMT_SHORTAGE_MAX        = ('AMT_SHORTAGE', 'max'),
    INST_AMT_SHORTAGE_MEAN       = ('AMT_SHORTAGE', 'mean'),
    INST_AMT_SHORTAGE_SUM        = ('AMT_SHORTAGE', 'sum'),

    # Tỷ lệ thanh toán
    INST_PAYMENT_RATIO_MEAN      = ('PAYMENT_RATIO', 'mean'),
    INST_PAYMENT_RATIO_MIN       = ('PAYMENT_RATIO', 'min'),

    # Số lần trả muộn
    INST_LATE_COUNT              = ('DAYS_LATE', lambda x: (x > 0).sum()),
    INST_SHORTAGE_COUNT          = ('AMT_SHORTAGE', lambda x: (x > 0).sum()),
).reset_index()

# Feature phái sinh
inst_agg['INST_LATE_RATE'] = inst_agg['INST_LATE_COUNT'] / inst_agg['INST_COUNT'].clip(lower=1)
inst_agg['INST_SHORTAGE_RATE'] = inst_agg['INST_SHORTAGE_COUNT'] / inst_agg['INST_COUNT'].clip(lower=1)

print(f"inst_agg shape: {inst_agg.shape}")
print(f"Features tạo từ installments: {inst_agg.shape[1] - 1}")
del installments
gc.collect()

**Các Feature quan trọng:**
- `INST_DAYS_LATE_MAX`: Lần trễ nhất là bao nhiêu ngày
- `INST_LATE_RATE`: Tỷ lệ kỳ trả trễ / tổng số kỳ
- `INST_PAYMENT_RATIO_MIN`: Lần trả ít nhất tính theo tỷ lệ


## 3.6. Credit_card_balance.csv - Thông tin dư nợ thẻ tín dụng

In [ ]:
# Aggregate credit_card_balance

credit_card['UTILIZATION'] = (
    credit_card['AMT_BALANCE'] / credit_card['AMT_CREDIT_LIMIT_ACTUAL'].clip(lower=1)
)

cc_agg = credit_card.groupby('SK_ID_CURR').agg(
    CC_MONTHS_COUNT              = ('MONTHS_BALANCE', 'count'),
    CC_AMT_BALANCE_MEAN          = ('AMT_BALANCE', 'mean'),
    CC_AMT_BALANCE_MAX           = ('AMT_BALANCE', 'max'),
    CC_AMT_DRAWING_MEAN          = ('AMT_DRAWINGS_CURRENT', 'mean'),
    CC_SK_DPD_MAX                = ('SK_DPD', 'max'),
    CC_SK_DPD_MEAN               = ('SK_DPD', 'mean'),
    CC_UTILIZATION_MEAN          = ('UTILIZATION', 'mean'),
    CC_UTILIZATION_MAX           = ('UTILIZATION', 'max'),
    CC_LIMIT_MEAN                = ('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
    CC_LATE_COUNT                = ('SK_DPD', lambda x: (x > 0).sum()),
).reset_index()

cc_agg['CC_LATE_RATE'] = cc_agg['CC_LATE_COUNT'] / cc_agg['CC_MONTHS_COUNT'].clip(lower=1)

print(f"cc_agg shape: {cc_agg.shape}")
print(f"Features tạo từ credit card balance: {cc_agg.shape[1] - 1}")
del credit_card
gc.collect()

** Các Features quan trọng:**
- `CC_UTILIZATION_MEAN`: Tỷ lệ sử dụng thẻ trung bình -> Nếu tỷ lệ cao ()> 80%), có thể tình hình tài chính tiền mặt đang không ổn
- `CC_SK_DPD_MAX`: Số ngày quá hạn thẻ tệ nhất
- `CC_LATE_RATE`: Tỷ lệ tháng có quá hạn thẻ


## 3.7. Tổng Hợp Features Trích Xuất từ Bảng Phụ


In [ ]:
# Tổng hợp features đã tạo từ mỗi bảng phụ

summary = {
    'bureau_agg': {
        'table': 'bureau.csv + bureau_balance.csv',
        'rows_raw': '1.7M + 27M',
        'features': list(bureau_agg.columns.drop('SK_ID_CURR')),
    },
    'prev_agg': {
        'table': 'previous_application.csv',
        'rows_raw': '1.7M',
        'features': list(prev_agg.columns.drop('SK_ID_CURR')),
    },
    'pos_agg': {
        'table': 'POS_CASH_balance.csv',
        'rows_raw': '10M',
        'features': list(pos_agg.columns.drop('SK_ID_CURR')),
    },
    'inst_agg': {
        'table': 'installments_payments.csv',
        'rows_raw': '13.6M',
        'features': list(inst_agg.columns.drop('SK_ID_CURR')),
    },
    'cc_agg': {
        'table': 'credit_card_balance.csv',
        'rows_raw': '3.8M',
        'features': list(cc_agg.columns.drop('SK_ID_CURR')),
    },
}

total_features = 0
for agg_name, info in summary.items():
    n = len(info['features'])
    total_features += n
    print(f"\n{'='*60}")
    print(f"  {agg_name}  <-  {info['table']}")
    print(f"  Raw rows: {info['rows_raw']}  ->  Features tạo ra: {n}")
    print(f"{'='*60}")
    for f in info['features']:
        print(f"    {f}")

print(f"  TỔNG: {total_features} features từ bảng phụ")

In [ ]:
#  xem tổng số feature mỗi bảng
for var_name in ['bureau_agg', 'prev_agg', 'pos_agg', 'inst_agg', 'cc_agg']:
    try:
        n = len(eval(var_name).columns) - 1  # trừ SK_ID_CURR
        print(f"{var_name:15s}: {n} features")
    except NameError:
        print(f"{var_name:15s}: chưa tạo")

**Tổng hợp ý nghĩa theo nhóm feature:**

###  Bureau (Lịch sử CIC) — 17 features
| Feature | Loại | Ý nghĩa kinh doanh |
|---------|------|-------------------|
| `BUREAU_LOAN_COUNT` | Count | Tổng số khoản vay từng có tại CIC |
| `BUREAU_ACTIVE_COUNT` | Count | Số khoản đang active → gánh nặng tín dụng hiện tại |
| `BUREAU_CLOSED_COUNT` | Count | Số khoản đã đóng → lịch sử trả xong nợ |
| `BUREAU_OVERDUE_MAX` | DPD | Số ngày quá hạn tệ nhất → **red flag mạnh nhất** |
| `BUREAU_OVERDUE_MEAN` | DPD | Trung bình ngày quá hạn qua các khoản vay |
| `BUREAU_AMT_OVERDUE_MAX` | Amount | Số tiền quá hạn lớn nhất từng có |
| `BUREAU_AMT_OVERDUE_MEAN` | Amount | Trung bình số tiền quá hạn |
| `BUREAU_AMT_CREDIT_SUM` | Amount | Tổng hạn mức tín dụng tại CIC |
| `BUREAU_AMT_CREDIT_MEAN` | Amount | Hạn mức trung bình mỗi khoản |
| `BUREAU_AMT_DEBT_SUM` | Amount | Tổng dư nợ hiện tại tại CIC |
| `BUREAU_AMT_DEBT_MEAN` | Amount | Dư nợ trung bình mỗi khoản |
| `BUREAU_DAYS_CREDIT_MAX` | Days | Khoản vay CIC gần nhất (ngày) |
| `BUREAU_DAYS_CREDIT_MEAN` | Days | Thời điểm vay trung bình |
| `BUREAU_DAYS_ENDDATE_MAX` | Days | Khoản vay kết thúc muộn nhất |
| `BUREAU_CNT_PROLONG_SUM` | Count | Tổng số lần gia hạn → dấu hiệu khó khăn tài chính |
| `BUREAU_ACTIVE_RATIO` | Ratio | Active / Total → tỷ lệ khoản chưa trả xong |
| `BUREAU_DEBT_RATIO` | Ratio | Dư nợ / Hạn mức → credit utilization tại CIC |

---

###  Previous Application (Lịch sử vay HC) — 15 features
| Feature | Loại | Ý nghĩa kinh doanh |
|---------|------|-------------------|
| `PREV_APP_COUNT` | Count | Tổng số lần từng apply vay tại HC |
| `PREV_APPROVED_COUNT` | Count | Số lần được duyệt |
| `PREV_REFUSED_COUNT` | Count | Số lần bị từ chối → **lịch sử xấu tại HC** |
| `PREV_CANCELED_COUNT` | Count | Số lần hủy (do khách hủy) |
| `PREV_AMT_APPLICATION_MEAN` | Amount | Số tiền yêu cầu trung bình |
| `PREV_AMT_APPLICATION_MAX` | Amount | Số tiền yêu cầu lớn nhất từng apply |
| `PREV_AMT_CREDIT_MEAN` | Amount | Số tiền được cấp trung bình |
| `PREV_AMT_CREDIT_MAX` | Amount | Số tiền được cấp lớn nhất |
| `PREV_AMT_DOWN_PAYMENT_MEAN` | Amount | Số tiền đặt cọc trung bình |
| `PREV_DAYS_DECISION_MAX` | Days | Lần quyết định gần nhất |
| `PREV_DAYS_DECISION_MEAN` | Days | Trung bình thời điểm quyết định |
| `PREV_RATE_DOWN_PAYMENT_MEAN` | Rate | Tỷ lệ đặt cọc → down payment cao = ít rủi ro |
| `PREV_RATE_INTEREST_MEAN` | Rate | Lãi suất trung bình các khoản trước |
| `PREV_APPROVAL_RATE` | Ratio | Approved / Total → tỷ lệ được duyệt |
| `PREV_CREDIT_TO_APP_RATIO` | Ratio | Cấp thực / Apply → mức độ bị cắt hạn mức |

---

###  POS CASH Balance (Hành vi POS/tiền mặt) — 9 features
| Feature | Loại | Ý nghĩa kinh doanh |
|---------|------|-------------------|
| `POS_MONTHS_COUNT` | Count | Số tháng có lịch sử POS |
| `POS_SK_DPD_MAX` | DPD | Số ngày quá hạn POS tệ nhất |
| `POS_SK_DPD_MEAN` | DPD | Trung bình ngày quá hạn POS |
| `POS_SK_DPD_DEF_MAX` | DPD | DPD định nghĩa tệ nhất (loại strict hơn) |
| `POS_SK_DPD_DEF_MEAN` | DPD | Trung bình DPD định nghĩa |
| `POS_COMPLETED_COUNT` | Count | Số khoản POS đã hoàn thành → lịch sử trả xong |
| `POS_ACTIVE_COUNT` | Count | Số khoản POS đang active |
| `POS_DPD_MONTHS_COUNT` | Count | Số tháng có ít nhất 1 ngày quá hạn |
| `POS_DPD_RATE` | Ratio | Tháng quá hạn / Tổng tháng → tần suất quá hạn |

---

###  Installments Payments (Hành vi trả góp — quan trọng nhất) — 13 features
| Feature | Loại | Ý nghĩa kinh doanh |
|---------|------|-------------------|
| `INST_COUNT` | Count | Tổng số kỳ trả góp trong lịch sử |
| `INST_DAYS_LATE_MAX` | Days | Lần trễ tệ nhất bao nhiêu ngày → **worst case behavior** |
| `INST_DAYS_LATE_MEAN` | Days | Trung bình số ngày trễ mỗi kỳ |
| `INST_DAYS_LATE_SUM` | Days | Tổng ngày trễ tích lũy toàn bộ lịch sử |
| `INST_AMT_SHORTAGE_MAX` | Amount | Lần trả thiếu tệ nhất (tiền mặt) |
| `INST_AMT_SHORTAGE_MEAN` | Amount | Trung bình số tiền trả thiếu |
| `INST_AMT_SHORTAGE_SUM` | Amount | Tổng tiền trả thiếu tích lũy |
| `INST_PAYMENT_RATIO_MEAN` | Ratio | Trả thực / Phải trả (TB) → ~1.0 là tốt, < 1 là thiếu |
| `INST_PAYMENT_RATIO_MIN` | Ratio | Tỷ lệ trả thấp nhất → lần tệ nhất trong lịch sử |
| `INST_LATE_COUNT` | Count | Số kỳ trả trễ trong toàn bộ lịch sử |
| `INST_SHORTAGE_COUNT` | Count | Số kỳ trả thiếu |
| `INST_LATE_RATE` | Ratio | Trễ / Tổng kỳ → **consistency trả nợ, signal mạnh nhất** |
| `INST_SHORTAGE_RATE` | Ratio | Thiếu / Tổng kỳ → tần suất trả không đủ |

---

###  Credit Card Balance (Thẻ tín dụng) — 11 features
| Feature | Loại | Ý nghĩa kinh doanh |
|---------|------|-------------------|
| `CC_MONTHS_COUNT` | Count | Số tháng có lịch sử thẻ tín dụng |
| `CC_AMT_BALANCE_MEAN` | Amount | Số dư thẻ trung bình → mức nợ thường xuyên |
| `CC_AMT_BALANCE_MAX` | Amount | Số dư thẻ cao nhất từng có |
| `CC_AMT_DRAWING_MEAN` | Amount | Số tiền rút/sử dụng thẻ trung bình/tháng |
| `CC_SK_DPD_MAX` | DPD | Số ngày quá hạn thẻ tệ nhất |
| `CC_SK_DPD_MEAN` | DPD | Trung bình ngày quá hạn thẻ |
| `CC_UTILIZATION_MEAN` | Ratio | Dư nợ / Hạn mức (TB) → > 80% = dấu hiệu căng thẳng TC |
| `CC_UTILIZATION_MAX` | Ratio | Utilization cao nhất từng có → worst case |
| `CC_LIMIT_MEAN` | Amount | Hạn mức thẻ trung bình → proxy creditworthiness |
| `CC_LATE_COUNT` | Count | Số tháng có quá hạn thẻ |
| `CC_LATE_RATE` | Ratio | Tháng quá hạn / Tổng tháng → tần suất quá hạn thẻ |

---

**Định hướng cho WOE/IV:**
Sau khi JOIN, tính IV cho tất cả features để xác định:
- Features nào thực sự predictive
- Features nào từ bảng phụ mạnh hơn so với bảng chính?
- Features nào khả năng bị data leakage



## 3.8. JOIN tất cả Aggregated Tables vào Bảng Chính

LEFT JOIN tất cả vào `df` theo `SK_ID_CURR`.

**LEFT JOIN vì**
- Không phải khách hàng nào cũng có lịch sử ở tất cả bảng phụ.
- LEFT JOIN giữ toàn bộ records từ bảng chính, những khách hàng không có
- lịch sử sẽ có NaN ở các cột từ bảng phụ -> đây cũng là thông tin có giá trị.


In [ ]:
# JOIN tất cả bảng phụ vào X_train và X_test
import gc

# 1. Gắn tạm SK_ID_CURR vào X_train, X_test dựa trên index để có key join
X_train['SK_ID_CURR'] = df.loc[X_train.index, 'SK_ID_CURR']
X_test['SK_ID_CURR'] = df.loc[X_test.index, 'SK_ID_CURR']

agg_tables = {
    'bureau_agg'  : bureau_agg,
    'prev_agg'    : prev_agg,
    'pos_agg'     : pos_agg,
    'inst_agg'    : inst_agg,
    'cc_agg'      : cc_agg,
}

shape_before = X_train.shape[1]

for name, agg_df in agg_tables.items():
    # Join trên X_train
    X_train = X_train.merge(agg_df, on='SK_ID_CURR', how='left')
    # Join trên X_test
    X_test = X_test.merge(agg_df, on='SK_ID_CURR', how='left')
    
    print(f"Sau merge {name}: X_train có {X_train.shape[1]} columns (+{X_train.shape[1] - shape_before})")
    shape_before = X_train.shape[1]

# 2. Xóa SK_ID_CURR sau khi join xong để không đưa vào model
X_train.drop(columns=['SK_ID_CURR'], inplace=True)
X_test.drop(columns=['SK_ID_CURR'], inplace=True)

print(f"\nShape cuối cùng - X_train: {X_train.shape} | X_test: {X_test.shape}")

# Giải phóng RAM
del bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg
gc.collect()


In [ ]:
# Xử lý NaN sinh ra từ JOIN
# Các cột mới từ bảng phụ (NaN = khách hàng không có lịch sử)
new_cols_from_join = [c for c in X_train.columns
                      if c.startswith(('BUREAU_', 'PREV_', 'POS_', 'INST_', 'CC_'))]

# Missing rate của các cột mới
missing_new = X_train[new_cols_from_join].isnull().mean().sort_values(ascending=False)
print("Missing rate của features từ bảng phụ (top 10):")
print(missing_new.head(10).round(3))

# Impute bằng 0 — ý nghĩa: "không có lịch sử" = "không có record" = 0
# (khác với bảng chính nơi 0 có thể không đúng)
for col in new_cols_from_join:
    X_train[col] = X_train[col].fillna(0)
    X_test[col]  = X_test[col].fillna(0)

print(f"\nSau impute X_train: missing còn lại = {X_train[new_cols_from_join].isnull().sum().sum()}")
print(f"Sau impute X_test:  missing còn lại = {X_test[new_cols_from_join].isnull().sum().sum()}")

In [ ]:
# Sanity check 
print("FINAL SANITY CHECK")

# 1. Missing values
total_missing = X_train.isnull().sum().sum()
test_missing  = X_test.isnull().sum().sum()
print(f"\n[1] Missing values")
print(f"    X_train: {total_missing}  {'ok' if total_missing == 0 else ' CẦN FIX'}")
print(f"    X_test:  {test_missing}   {'ok' if test_missing == 0 else ' CẦN FIX'}")

# 2. Shape hợp lý
print(f"\n[2] Shape")
print(f"    X_train: {X_train.shape}  {'ok' if X_train.shape[0] > 200000 else 'CẦN FIX'}")
print(f"    X_test:  {X_test.shape}")

# 3. Bad rate sau split
br_train = y_train.mean()
br_test  = y_test.mean()
print(f"\n[3] Bad rate (phải ~8.07% cả 2)")
print(f"    Train: {br_train*100:.2f}%")
print(f"    Test:  {br_test*100:.2f}%  (OOT — bad rate khác train là bình thường)")
if abs(br_test - br_train) > 0.05:
    print("    ⚠ Chênh lệch bad rate > 5% giữa train/test — kiểm tra lại split logic")
else:
    print("    ok")

# 4. Inf values
inf_count = np.isinf(X_train.select_dtypes('float64')).sum().sum()
print(f"\n[4] Inf values trong X_train: {inf_count}  {'ok' if inf_count == 0 else 'CẦN FIX'}")

# 5. Duplicate columns
dup = X_train.columns[X_train.columns.duplicated()].tolist()
print(f"\n[5] Duplicate columns: {dup if dup else 'None'}  {'ok' if not dup else 'CẦN FIX'}")

# 6. Feature groups
aux_in_train = [c for c in X_train.columns if c.startswith(('BUREAU_','PREV_','POS_','INST_','CC_'))]
print(f"\n[6] Features từ bảng phụ trong X_train: {len(aux_in_train)}  {'ok' if len(aux_in_train) > 50 else 'CẦN FIX'}")

# 7. EXT_SOURCE có trong features không
ext_check = [c for c in X_train.columns if 'EXT_SOURCE' in c or c in ['EXT_MEAN','EXT_MIN','EXT_SOURCE_MEAN']]
print(f"\n[7] EXT_SOURCE features: {ext_check}")

# 8. Summary
print(f"\n{'=' * 60}")
all_ok = (total_missing == 0 and test_missing == 0 and inf_count == 0 and not dup)
print(f"  STATUS: {' DATA SẴN SÀNG' if all_ok else ' CÒN VẤN ĐỀ - XEM LẠI Ở TRÊN'}")

# 4. WOE & IV Analysis
WOE và IV phục vụ 2 mục đích:

1. Tính toán IV  để lọc features
2. WOE Transform: Chuyển giá trị raw -> WOE cho Logistic Regression


**Công thức gốc:**
WOE_i = ln( %GOOD trong bin_i / %BAD trong bin_i ) = ln( (Good_i / Total_Good) / (Bad_i / Total_Bad) )

IV = Σ (%Good_i - %Bad_i) × WOE_i (tổng qua tất cả bins)

**Ngưỡng IV theo Naeem Siddiqi:**
| IV | Đánh giá |
|----|---------|
| < 0.02 | Useless — không có signal |
| 0.02 – 0.1 | Weak — yếu, cân nhắc |
| 0.1 – 0.3 | Medium — tốt |
| 0.3 – 0.5 | Strong — rất tốt |
| > 0.5 | Suspicious — nghi leakage |


## 4.1. Định nghĩa hàm WOE/IV3 hàm cốt lõi:

- `calc_woe_iv(df, feature, target)` → tính WOE từng bin + IV của 1 feature
- `calc_iv_all(df, features, target)` → tính IV cho toàn bộ feature list
- `woe_transform(df, feature, bins_df)` → transform raw values sang WOE values


In [ ]:
# Hàm WOE/IV
import numpy as np
import pandas as pd

def calc_woe_iv(df, feature, target, n_bins=10):
    total_good = (df[target] == 0).sum()
    total_bad  = (df[target] == 1).sum()

    temp = df[[feature, target]].copy()

    # Binning
    if temp[feature].dtype == 'object' or temp[feature].nunique() <= n_bins:
        temp['bin'] = temp[feature].astype(str)
    else:
        try:
            temp['bin'] = pd.qcut(temp[feature], q=n_bins, duplicates='drop')
        except Exception:
            temp['bin'] = pd.cut(temp[feature], bins=n_bins, duplicates='drop')

    # Aggregate theo bin
    grp = temp.groupby('bin', observed=True)[target].agg(['sum', 'count'])
    grp.columns = ['bad', 'total']
    grp['good'] = grp['total'] - grp['bad']

    # LƯU Ý BUG FIX 1: Ngoặc đúng chuẩn toán học để tính %
    grp['pct_good'] = (grp['good'] + 0.5) / total_good
    grp['pct_bad']  = (grp['bad']  + 0.5) / total_bad

    grp['woe'] = np.log(grp['pct_good'] / grp['pct_bad'])
    grp['iv']  = (grp['pct_good'] - grp['pct_bad']) * grp['woe']

    iv_total = grp['iv'].sum()
    grp['feature'] = feature
    grp['bad_rate'] = grp['bad'] / grp['total']

    return iv_total, grp.reset_index().rename(columns={'bin': 'bin_label'})


def calc_iv_all(df, features, target, n_bins=10):
    results = []
    for i, feat in enumerate(features):
        if (i + 1) % 50 == 0:
            print(f"  Đã xử lý {i+1}/{len(features)} features...")
        try:
            iv_val, _ = calc_woe_iv(df, feat, target, n_bins=n_bins)
            results.append({'Variable': feat, 'IV': round(iv_val, 6)})
        except Exception:
            results.append({'Variable': feat, 'IV': 0.0})

    iv_df = pd.DataFrame(results).sort_values('IV', ascending=False).reset_index(drop=True)
    return iv_df


def woe_transform(df, feature, bin_df):
    col = df[feature].copy()
    result = pd.Series(0.0, index=col.index)
    
    if col.dtype == 'object' or col.nunique() <= 10:
        # String/Categorical mapping
        woe_map = dict(zip(bin_df['bin_label'].astype(str), bin_df['woe']))
        result = col.astype(str).map(woe_map).fillna(0.0)
    else:
        # Robust Interval mapping (tránh lỗi pd.cut mismatch)
        for _, row in bin_df.iterrows():
            bin_obj = row['bin_label']
            woe_val = row['woe']
            
            if hasattr(bin_obj, 'left') and hasattr(bin_obj, 'right'):
                # Nó là pandas Interval
                mask = (col > bin_obj.left) & (col <= bin_obj.right)
                result.loc[mask] = woe_val
            else:
                # Fallback nếu nó là string representation của interval
                if isinstance(bin_obj, str) and '(' in bin_obj and ']' in bin_obj:
                    try:
                        left, right = map(float, bin_obj.strip('()[]').split(','))
                        mask = (col > left) & (col <= right)
                        result.loc[mask] = woe_val
                    except:
                        mask = (col.astype(str) == str(bin_obj))
                        result.loc[mask] = woe_val
                else:
                    mask = (col.astype(str) == str(bin_obj))
                    result.loc[mask] = woe_val

    return result.astype(float)

print(" Đã định nghĩa 3 hàm: calc_woe_iv | calc_iv_all | woe_transform (Robust)")


## 4.2. Chuẩn bị data và tính IV


In [ ]:
# Chuẩn bị df_woe

df_woe = X_train.copy()
df_woe['TARGET'] = y_train.values

# Loại các cột không phù hợp với WOE binning
# (_ENC là label encoded cats, _IS_MISSING là binary flags — để riêng)
exclude_iv = [c for c in df_woe.columns
              if c.endswith('_ENC') or c.endswith('_IS_MISSING') or c == 'TARGET']
feature_for_iv = [c for c in df_woe.columns if c not in exclude_iv]

print(f"df_woe shape: {df_woe.shape}")
print(f"Bad rate: {df_woe['TARGET'].mean()*100:.2f}%")
print(f"Features đưa vào tính IV: {len(feature_for_iv)}")

## 4.3. Tính IV toàn bộ features


In [ ]:
# Tính IV

print(f"Tính IV cho {len(feature_for_iv)} features...")

iv_df = calc_iv_all(df_woe, feature_for_iv, target='TARGET', n_bins=10)

# Phân loại theo ngưỡng
def classify_iv(iv):
    if iv < 0.02:   return 'Useless'
    elif iv < 0.1:  return 'Weak'
    elif iv < 0.3:  return 'Medium'
    elif iv < 0.5:  return 'Strong'
    else:           return 'Suspicious'

iv_df['Category'] = iv_df['IV'].apply(classify_iv)

# Summary
summary_iv = iv_df['Category'].value_counts()
print("\nPhân loại theo IV:")
for cat in ['Suspicious', 'Strong', 'Medium', 'Weak', 'Useless']:
    n = summary_iv.get(cat, 0)
    print(f"  {cat:12s}: {n:3d} features")

print(f"\nTop 20 features:")
print(iv_df.head(20).to_string(index=False))

In [ ]:
# Chuẩn bị IV dataframe sạch để visualize và feature selection

iv_df_clean = iv_df.copy()

# Nếu có duplicate feature name thì giữ bản IV cao nhất
iv_df_clean = (
    iv_df_clean
    .sort_values("IV", ascending=False)
    .drop_duplicates(subset=["Variable"], keep="first")
    .reset_index(drop=True)
)

print(f"Tổng số features sau khi loại duplicate: {len(iv_df_clean)}")
print("\nTop 10 features theo IV:")
print(iv_df_clean.head(10)[["Variable", "IV", "Category"]].to_string(index=False))


## 4.4. Visualize IV Ranking


In [ ]:
color_map = {
    'Suspicious': '#9C27B0',
    'Strong':     '#F44336',
    'Medium':     '#FF9800',
    'Weak':       '#FFC107',
    'Useless':    '#BDBDBD'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Chart: Top 30 features
top30 = iv_df_clean.head(30)
colors = top30['Category'].map(color_map)
axes[0].barh(top30['Variable'][::-1], top30['IV'][::-1], color=colors[::-1])
for thresh, color, label in [(0.02, 'gray', 'Useless (0.02)'),
                              (0.1,  '#FFC107', 'Weak (0.1)'),
                              (0.3,  '#FF9800', 'Medium (0.3)'),
                              (0.5,  '#F44336', 'Strong (0.5)')]:
    axes[0].axvline(x=thresh, color=color, linestyle='--', alpha=0.7, label=label)
axes[0].set_title('Top 30 Features theo IV', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Information Value')
axes[0].legend(fontsize=8)

# Chart: Phân bổ iV
cat_order  = ['Suspicious', 'Strong', 'Medium', 'Weak', 'Useless']
cat_counts = [summary_iv.get(c, 0) for c in cat_order]
cat_colors = [color_map[c] for c in cat_order]
axes[1].bar(cat_order, cat_counts, color=cat_colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Phân bố IV Categories)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Số features')
for i, v in enumerate(cat_counts):
    if v > 0:
        axes[1].text(i, v + 0.3, str(v), ha='center', fontweight='bold', fontsize=11)

plt.suptitle('IV Analysis — Feature Ranking cho Credit Scorecard',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.5. Feature Selection theo IV


In [ ]:
# Feature Selection theo IV (từ iv_df_clean)

# Loại Useless (IV < 0.02) từ bản đã loại duplicate
selected_features_iv = iv_df_clean[iv_df_clean['IV'] >= 0.02]['Variable'].tolist()
selected_features_iv = [f for f in selected_features_iv if f in X_train.columns]

print(f"\nFeatures ban đầu (raw):      {len(feature_for_iv)}")
print(f"Sau khi loại duplicate:           {len(iv_df_clean)}")
print(f"Sau khi filter IV≥0.02:           {len(selected_features_iv)}")
print(f"Đã loại (Useless):            {len(iv_df_clean) - len(selected_features_iv)}")
print(f"\nTop 10 features cuối cùng:")
print(iv_df_clean.head(10)[['Variable', 'IV', 'Category']].to_string(index=False))

**Nhận xét:**
- Đối với mô hình Logistic Regression + WOE dùng `selected_features_iv`.
- Đối với LightGBM sẽ dùng X_train raw — có thể giữ cả features IV thấp vì tree-based model tự tìm interaction, không bị giới hạn bởi tuyến tính.


## 4.6. WOE Bins chi tiết cho Top Features


In [ ]:
# Tính WOE bins cho top features và visualize

# Top features từ Medium và Strong (dùng iv_df_clean)
top_features_plot = iv_df_clean[
    iv_df_clean['Category'].isin(['Strong', 'Medium'])
]['Variable'].head(9).tolist()
top_features_plot = [f for f in top_features_plot if f in X_train.columns]

print(f"Visualize WOE bins cho: {top_features_plot}")

# Tính WOE bins
bins_top = {}
for feat in top_features_plot:
    _, bin_df = calc_woe_iv(df_woe, feat, 'TARGET', n_bins=10)
    bins_top[feat] = bin_df

# Plot
n_plot = min(6, len(top_features_plot))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features_plot[:n_plot]):
    ax   = axes[i]
    ax2  = ax.twinx()
    bdf  = bins_top[feat]

    ax.bar(range(len(bdf)), bdf['total'], alpha=0.35,
           color='#90CAF9', label='Total count')
    ax.bar(range(len(bdf)), bdf['bad'], alpha=0.6,
           color='#EF9A9A', label='Bad count')
    ax2.plot(range(len(bdf)), bdf['woe'], 'ro-',
             linewidth=2, markersize=7, label='WOE')
    ax2.axhline(0, color='black', linestyle='--', linewidth=0.8)

    ax.set_xticks(range(len(bdf)))
    ax.set_xticklabels(
        [str(b)[:12] for b in bdf['bin_label']],
        rotation=45, ha='right', fontsize=7
    )
    ax.set_ylabel('Count', color='#1565C0', fontsize=9)
    ax2.set_ylabel('WOE', color='red', fontsize=9)

    iv_val = iv_df_clean[iv_df_clean['Variable'] == feat]['IV'].values[0]
    cat    = iv_df_clean[iv_df_clean['Variable'] == feat]['Category'].values[0]
    ax.set_title(f'{feat}\nIV={iv_val:.4f} [{cat}]', fontsize=9, fontweight='bold')

# Tắt axis thừa
for j in range(n_plot, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'WOE Bins — Top Features\n'
    'WOE > 0: ít rủi ro | WOE < 0: nhiều rủi ro | Monotonic tốt cho Logistic Regression',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

**Ghi chú cho WOE plot:**

- **Bar xanh** = tổng số records trong bin | **Bar đỏ** = số BAD records
- **Đường đỏ** = WOE của từng bin
- **Monotonic** (đường WOE đi đều từ âm lên dương hoặc ngược lại) -> tốt cho LR
- **Bin quá nhỏ** (bar rất thấp) -> WOE không ổn định, cân nhắc merge bin đó


## 4.7. WOE Transform toàn bộ features
- Sau khi biết WOE từng bin, transform toàn bộ X_train và X_test.
- Nguyên tắc chống leakage: **bins chỉ được tính từ X_train**, sau đó apply lên cả 2.


In [ ]:
# Tính WOE bins cho toàn bộ selected features

print(f"Tính WOE bins cho {len(selected_features_iv)} features...")

bins_all = {}
for i, feat in enumerate(selected_features_iv):
    if (i + 1) % 30 == 0:
        print(f"  {i+1}/{len(selected_features_iv)}...")
    try:
        _, bin_df = calc_woe_iv(df_woe, feat, 'TARGET', n_bins=10)
        bins_all[feat] = bin_df
    except Exception as e:
        print(f"  ⚠ Skip {feat}: {e}")

print(f"\nĐã tính bins cho {len(bins_all)} features")

# WOE transform X_train
print("\nTransform X_train...")
X_train_woe = pd.DataFrame(index=X_train.index)
for feat in bins_all:
    X_train_woe[feat + '_woe'] = woe_transform(X_train, feat, bins_all[feat])

# WOE transform X_test (dùng bins từ train)
print("Transform X_test...")
X_test_woe = pd.DataFrame(index=X_test.index)
for feat in bins_all:
    X_test_woe[feat + '_woe'] = woe_transform(X_test, feat, bins_all[feat])

print(f"\nX_train_woe shape: {X_train_woe.shape}")
print(f"X_test_woe shape:  {X_test_woe.shape}")
print(f"Missing X_train_woe: {X_train_woe.isnull().sum().sum()}")
print(f"\nSample giá trị WOE (3 rows đầu, 4 cols đầu):")
print(X_train_woe.iloc[:3, :4].round(4))

# Append _IS_MISSING flags vào WOE data (không WOE-transform, dùng trực tiếp vì đã binary 0/1)
is_missing_cols = [c for c in X_train.columns if c.endswith('_IS_MISSING')]

for col in is_missing_cols:
    X_train_woe[col] = X_train[col].values
    X_test_woe[col]  = X_test[col].values

print(f"\nĐã thêm {len(is_missing_cols)} IS_MISSING flags vào WOE data.")
print(f"X_train_woe shape sau khi thêm: {X_train_woe.shape}")


In [ ]:
# Sanity Check WOE

print("WOE/IV SUMMARY")
print(f"Features đưa vào IV (raw):    {len(feature_for_iv)}")
print(f"Sau loại duplicate:           {len(iv_df_clean)}")
print(f"Sau filter IV≥0.02:           {len(selected_features_iv)}")
print(f"WOE-transformed features:     {X_train_woe.shape[1]}")
print(f"Missing trong WOE data:       {X_train_woe.isnull().sum().sum()}")

print(f"\nTop 10 features theo IV (sau clean):")
print(iv_df_clean.head(10)[['Variable', 'IV', 'Category']].to_string(index=False))

# Kiểm tra WOE values hợp lý
woe_min = X_train_woe.min().min()
woe_max = X_train_woe.max().max()
print(f"\nWOE range: [{woe_min:.3f}, {woe_max:.3f}]")
print(f"Hợp lý nếu range trong khoảng [-3, 3]  {'ok' if -4 < woe_min and woe_max < 4 else 'Kiểm tra lại'}")

## 5.1. Train Logistic Regression

Dùng `LogisticRegression` với regularization L1


In [ ]:
# 5.1. Train Logistic Regression với Lasso + VIF Feature Selection
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Fill NaN còn lại trong WOE data (nếu có)
X_train_woe_filled = X_train_woe.fillna(0)
X_test_woe_filled  = X_test_woe.fillna(0)

# 2. L1/Lasso: lấy shortlist biến mạnh nhất, giới hạn số lượng features trong scorecard
print("L1 Regularization: shortlist top 20 biến mạnh nhất cho scorecard...")
lasso_selector = LogisticRegression(
    penalty='l1',
    C=0.01,
    solver='liblinear',
    max_iter=1000,
    random_state=42
)
lasso_selector.fit(X_train_woe_filled, y_train)

lasso_coef = pd.Series(lasso_selector.coef_[0], index=X_train_woe_filled.columns)
nonzero_coef = lasso_coef[lasso_coef != 0].abs().sort_values(ascending=False)
selected_lasso = nonzero_coef.head(20).index.tolist()
print(f"Features non-zero bởi Lasso: {len(nonzero_coef)}")
print(f"Giữ top {len(selected_lasso)} features trước VIF")

# 3. VIF check: loại đa cộng tuyến cao trên shortlist WOE features
def calculate_vif_table(X):
    rows = []
    X_num = X.astype(float)
    for col in X_num.columns:
        y_col = X_num[col].values
        X_other = X_num.drop(columns=[col]).values
        if X_other.shape[1] == 0 or np.nanstd(y_col) == 0:
            vif = 1.0
        else:
            r2 = LinearRegression().fit(X_other, y_col).score(X_other, y_col)
            vif = np.inf if r2 >= 0.999999 else 1.0 / (1.0 - r2)
        rows.append({'Feature': col, 'VIF': vif})
    return pd.DataFrame(rows).sort_values('VIF', ascending=False).reset_index(drop=True)

selected_features = selected_lasso.copy()
vif_history = []
while len(selected_features) > 2:
    vif_table = calculate_vif_table(X_train_woe_filled[selected_features])
    max_vif = vif_table.iloc[0]['VIF']
    if max_vif <= 10:
        break
    drop_feature = vif_table.iloc[0]['Feature']
    selected_features.remove(drop_feature)
    vif_history.append((drop_feature, max_vif))

vif_table_final = calculate_vif_table(X_train_woe_filled[selected_features])
print(f"Loại {len(vif_history)} features do VIF > 10")
if vif_history:
    print(pd.DataFrame(vif_history, columns=['Dropped_Feature', 'VIF']).to_string(index=False))
print("\nVIF sau xử lý:")
print(vif_table_final.to_string(index=False, float_format='{:.2f}'.format))

X_train_woe_sel = X_train_woe_filled[selected_features]
X_test_woe_sel  = X_test_woe_filled[selected_features]

# 4. Train LR cuối cùng không dùng class_weight để probability/score phản ánh PD tự nhiên
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)
lr_model.fit(X_train_woe_sel, y_train)

# Predict probability
# Lưu ý: đây là raw PD estimate từ model không re-weight class; phù hợp hơn cho score scaling.
y_train_prob = lr_model.predict_proba(X_train_woe_sel)[:, 1]
y_test_prob  = lr_model.predict_proba(X_test_woe_sel)[:, 1]

# Tính AUC và GINI
train_auc = roc_auc_score(y_train, y_train_prob)
test_auc  = roc_auc_score(y_test,  y_test_prob)
train_gini = 2 * train_auc - 1
test_gini  = 2 * test_auc  - 1

print("\nLOGISTIC REGRESSION SCORECARD — KẾT QUẢ")
print(f"Train AUC:  {train_auc:.4f}  |  GINI: {train_gini:.4f}")
print(f"Test  AUC:  {test_auc:.4f}  |  GINI: {test_gini:.4f}")
print(f"\nFeatures trong scorecard cuối cùng ({len(selected_features)}):")
print(selected_features)


## 5.2. Kiểm tra Hệ Số Logistic Regression

Sau khi train, kiểm tra hệ số (coefficients) của từng feature.

**Quy tắc dấu trong WOE Scorecard:**
- WOE > 0 → bin này có tỷ lệ GOOD cao -> Đồng nghĩa ít rủi ro -> coefficient âm (giảm log-odds default)
- WOE < 0 → bin này có tỷ lệ BAD cao -> Đồng nghĩa nhiều rủi ro -> coefficient dương

→ Hệ số LR nên có dấu **âm** với features tốt (ví dụ EXT_SOURCE cao -> ít rủi ro -> coefficient âm)
→ Hệ số **dương** với features rủi ro (ví dụ DPD cao → nhiều rủi ro → coefficient dương)

Trường hơp dấu bị ngược -> kiểm tra lại multicollinearity hoặc WOE transform


In [ ]:
# Kiểm tra hệ số

# Tạo DataFrame hệ số
coef_df = pd.DataFrame({
    'Feature'    : X_train_woe_sel.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).reset_index(drop=True)

# Thêm IV tương ứng
coef_df = coef_df.merge(
    iv_df_clean[['Variable', 'IV', 'Category']].rename(
        columns={'Variable': 'Feature_base'}),
    left_on=coef_df['Feature'].str.replace('_woe', ''),
    right_on='Feature_base',
    how='left'
).drop(columns=['key_0', 'Feature_base'], errors='ignore')

print("Top 20 features theo |Coefficient|:")
print(coef_df.head(20)[['Feature', 'Coefficient', 'IV']].to_string(index=False))

# Kiểm tra dấu hệ số
n_positive = (coef_df['Coefficient'] > 0).sum()
n_negative = (coef_df['Coefficient'] < 0).sum()
print(f"\nHệ số dương (rủi ro cao hơn khi WOE tăng): {n_positive}")
print(f"Hệ số âm  (an toàn hơn khi WOE tăng):     {n_negative}")
print(f"\nBias (intercept): {lr_model.intercept_[0]:.4f}")

## 5.3. Visualize Coefficients


In [ ]:
# Visualize hệ số

top_n = 20
top_coef = coef_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#EF5350' if c > 0 else '#42A5F5' for c in top_coef['Coefficient']]
bars = ax.barh(
    top_coef['Feature'][::-1],
    top_coef['Coefficient'][::-1],
    color=colors[::-1],
    edgecolor='white'
)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Coefficient (Logistic Regression)', fontsize=11)
ax.set_title(
    f'Top {top_n} Features — Hệ số Logistic Regression\n'
    'Dương : rủi ro tăng | Âm : rủi ro giảm',
    fontsize=12, fontweight='bold'
)

# Thêm giá trị lên bar
for bar, val in zip(bars[::-1], top_coef['Coefficient']):
    ax.text(
        val + (0.002 if val >= 0 else -0.002),
        bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}', va='center',
        ha='left' if val >= 0 else 'right', fontsize=8
    )

plt.tight_layout()
plt.show()

## 5.4. Output của kết quả mô hình LR
**3 lớp** phục vụ các đối tượng khác nhau:


| Lớp | Output | Đối tượng sử dụng| Mục đích |
|-----|--------|---------|----------|
| 1 — Probability | P(default) ∈ [0,1] | Data scientist | Đánh giá model (GINI, AUC, Calibration) |
| 2 — Credit Score | 300–850 điểm | Risk manager | Cutoff, pricing, limit setting |
| 3 — Risk Grade | A / B / C / D / E | Credit officer, ban lãnh đạo | Quyết định, báo cáo, tuân thủ |

### 5.4.1. Lớp đánh giá mô hình


In [ ]:
# AUC/GINI và KS Statistic đo khả năng phân tách Good/Bad mà không cần chọn cutoff
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             classification_report, confusion_matrix,
                             average_precision_score)
from scipy import stats

# KS Statistic
fpr, tpr, thresholds_roc = roc_curve(y_test, y_test_prob)
ks_stat = (tpr - fpr).max()
ks_thresh = thresholds_roc[(tpr - fpr).argmax()]

print(f"  AUC:          {test_auc:.4f}")
print(f"  GINI:         {test_gini:.4f}  (Benchmark: 0.49)")
print(f"  KS Statistic: {ks_stat:.4f}")
print(f"  KS tại prob threshold: {ks_thresh:.4f}")

In [ ]:
# ROC + Precision-Recall Curve ──────────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
axes[0].plot(fpr, tpr, color='#1565C0', linewidth=2.5,
             label=f'LR Scorecard (AUC={test_auc:.4f} | GINI={test_gini:.4f})')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#1565C0')
axes[0].plot([0,1],[0,1],'k--', linewidth=1, label='Random baseline')
# KS annotation
ks_idx = (tpr - fpr).argmax()
axes[0].annotate(f'KS={ks_stat:.3f}',
                 xy=(fpr[ks_idx], tpr[ks_idx]),
                 xytext=(fpr[ks_idx]+0.1, tpr[ks_idx]-0.1),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 color='red', fontweight='bold')
axes[0].plot([fpr[ks_idx], fpr[ks_idx]], [fpr[ks_idx], tpr[ks_idx]],
             'r--', linewidth=1.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve + KS Statistic', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_test_prob)
ap_score = average_precision_score(y_test, y_test_prob)
baseline_pr = y_test.mean()
axes[1].plot(recall, precision, color='#E53935', linewidth=2.5,
             label=f'LR Scorecard (AP={ap_score:.4f})')
axes[1].axhline(y=baseline_pr, color='gray', linestyle='--', linewidth=1.5,
                label=f'Baseline (Bad rate={baseline_pr:.3f})')
axes[1].fill_between(recall, precision, alpha=0.08, color='#E53935')
axes[1].set_xlabel('Recall (= Sensitivity)')
axes[1].set_ylabel('Precision (= PPV)')
axes[1].set_title('Precision-Recall Curve',
                  fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4.2. Lớp score scaling (Factor/Offset/PDO)

**PDO Framework (Points to Double Odds):**

Factor = PDO / ln(2) Offset = Base_Score - Factor × ln(Base_Odds) Score = Offset - Factor × ln(P_bad / P_good)

**Tham số:**
- PDO = 20: Mỗi khi odds tăng gấp đôi, score giảm 20 điểm
- Base_Score = 600: Điểm chuẩn reference point
- Base_Odds = 50: Tại điểm 600, tỷ lệ GOOD:BAD = 50:1

Score cao = ít rủi ro. Score thấp = rủi ro cao.


In [ ]:
# Tham số scaling
PDO        = 20
BASE_SCORE = 600
BASE_ODDS  = 50   # GOOD:BAD = 50:1 tại điểm 600

# Tính Factor và Offset
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)

print(f"Scaling Parameters:")
print(f"  PDO        = {PDO}")
print(f"  Base Score = {BASE_SCORE}")
print(f"  Base Odds  = {BASE_ODDS}:1")
print(f"  Factor     = {FACTOR:.4f}")
print(f"  Offset     = {OFFSET:.4f}")

# Convert probability → score
def prob_to_score(prob_bad, factor=FACTOR, offset=OFFSET):
    """
    Chuyển xác suất vỡ nợ thành credit score.
    Score cao = ít rủi ro (GOOD).
    """
    prob_bad  = np.clip(prob_bad, 1e-7, 1 - 1e-7)  # tránh log(0)
    prob_good = 1 - prob_bad
    log_odds  = np.log(prob_bad / prob_good)         # log(odds_bad)
    score     = offset - factor * log_odds            # score cao = good
    return np.round(score).astype(int)

# Tính score cho train và test
train_scores = prob_to_score(y_train_prob)
test_scores  = prob_to_score(y_test_prob)

print(f"\nScore distribution (Test):")
print(f"  Min:  {test_scores.min()}")
print(f"  Max:  {test_scores.max()}")
print(f"  Mean: {test_scores.mean():.1f}")
print(f"  Std:  {test_scores.std():.1f}")

In [ ]:
# Visualize Score Distribution

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Test set: Good vs Bad score distribution
good_scores = test_scores[y_test.values == 0]
bad_scores  = test_scores[y_test.values == 1]

axes[0].hist(good_scores, bins=50, alpha=0.6, color='#42A5F5',
             density=True, label=f'GOOD (n={len(good_scores):,})')
axes[0].hist(bad_scores,  bins=50, alpha=0.6, color='#EF5350',
             density=True, label=f'BAD  (n={len(bad_scores):,})')
axes[0].axvline(np.median(good_scores), color='#1565C0', linestyle='--',
                linewidth=2, label=f'Median GOOD: {np.median(good_scores):.0f}')
axes[0].axvline(np.median(bad_scores),  color='#B71C1C', linestyle='--',
                linewidth=2, label=f'Median BAD: {np.median(bad_scores):.0f}')
axes[0].set_xlabel('Credit Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution — GOOD vs BAD', fontsize=12, fontweight='bold')
axes[0].legend()

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
axes[1].plot(fpr, tpr, color='#1565C0', linewidth=2.5,
             label=f'LR Scorecard (AUC = {test_auc:.4f}, GINI = {test_gini:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
axes[1].axhline(y=0.49/2 + 0.5, color='red', linestyle='--', alpha=0.7,
                label=f'Benchmark GINI = 0.49')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Logistic Regression', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Track 1: Logistic Regression Scorecard — Evaluation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.4.3.  Chuyển Score -> Risk Grade

Chuyển sang ngôn ngữ business 

Sau khi có credit score --> chuyển điểm thành các nhóm risk grade A/B/C/D/E để dễ dùng trong báo cáo và quyết định tín dụng.

Với bộ tham số PDO, score trong khoảng 472-633.

Nên thay vì dùng ngưỡng cố định:

- A >= 750
- B >= 650
- C >= 550
- D >= 450

thì các grade sẽ bị lệch: phần lớn khách hàng chỉ rơi vào một vài nhóm, làm cho risk grade không còn hữu ích để so sánh bad rate hoặc đặt cutoff.

Trong thực tế, ngưỡng grade thường được hiệu chỉnh theo:

1. Phân phối score của portfolio hiện tại.
2. Bad rate quan sát được trong từng score band.
3. Khẩu vị rủi ro và chính sách approval của tổ chức.
4. Yêu cầu về tỷ lệ approve/reject hoặc manual review.

Phạm vi đồ án dùng cách chia theo phân vị score:

- Top 20% score cao nhất -> Grade A
- 20% tiếp theo -> Grade B
- 20% giữa -> Grade C
- 20% tiếp theo -> Grade D
- Bottom 20% score thấp nhất -> Grade E

Cách này giúp các grade có quy mô tương đối cân bằng, dễ kiểm tra bad rate giữa các nhóm và phù hợp hơn cho mục tiêu phân tích trong đồ án. Sau khi chia grade, cần kiểm tra lại bad rate có tăng dần từ A đến E hay không. Nếu bad rate tách biệt tốt, risk grade có ý nghĩa về mặt nghiệp vụ.


In [ ]:
# Tính ngưỡng theo phân vị của test_scores
q20 = int(np.percentile(test_scores, 20))
q40 = int(np.percentile(test_scores, 40))
q60 = int(np.percentile(test_scores, 60))
q80 = int(np.percentile(test_scores, 80))

print("Ngưỡng Grade theo phân phối thực tế (test set):")
print(f"  A (top 20%):     score >= {q80}")
print(f"  B (60–80%):      {q60} – {q80}")
print(f"  C (40–60%):      {q40} – {q60}")
print(f"  D (20–40%):      {q20} – {q40}")
print(f"  E (bottom 20%):  score < {q20}")

GRADE_LABELS = {
    'A': 'A — Xuất sắc  (Rủi ro rất thấp)',
    'B': 'B — Tốt       (Rủi ro thấp)',
    'C': 'C — Trung bình (Rủi ro trung bình)',
    'D': 'D — Thấp      (Rủi ro cao)',
    'E': 'E — Từ chối   (Rủi ro rất cao)',
}

def assign_grade(score):
    if   score >= q80: return 'A'
    elif score >= q60: return 'B'
    elif score >= q40: return 'C'
    elif score >= q20: return 'D'
    else:              return 'E'

# Gán grade cho test set
test_grades = pd.Series(test_scores).apply(assign_grade)

# Tạo bảng phân tích
grade_analysis = pd.DataFrame({
    'Score'     : test_scores,
    'Grade'     : test_grades.values,
    'Actual_Bad': y_test.values
})

grade_summary = grade_analysis.groupby('Grade').agg(
    Count      = ('Score', 'count'),
    Bad_Count  = ('Actual_Bad', 'sum'),
    Score_Mean = ('Score', 'mean'),
    Score_Min  = ('Score', 'min'),
    Score_Max  = ('Score', 'max'),
).reset_index()

grade_summary['Bad_Rate']        = grade_summary['Bad_Count'] / grade_summary['Count']
grade_summary['Pct_Population']  = grade_summary['Count'] / grade_summary['Count'].sum()
grade_summary['Grade_Label']     = grade_summary['Grade'].map(GRADE_LABELS)

print("\n" + "=" * 70)
print("RISK GRADE ANALYSIS")
print("=" * 70)
cols = ['Grade_Label', 'Count', 'Pct_Population', 'Bad_Rate', 'Score_Mean']
print(grade_summary[cols].to_string(index=False, float_format='{:.4f}'.format))
print(f"\nBaseline bad rate: {y_test.mean()*100:.2f}%")

In [ ]:
# Visualize phân phối grade

grade_order  = ['A', 'B', 'C', 'D', 'E']
grade_colors = ['#1565C0', '#42A5F5', '#FFC107', '#FF7043', '#B71C1C']

# Re-index để đảm bảo đủ 5 grades
gs = grade_summary.set_index('Grade').reindex(grade_order).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Chart 1: Phân phối theo grade
axes[0].bar(gs['Grade'], gs['Pct_Population'] * 100,
            color=grade_colors, edgecolor='white', linewidth=1.5)
for i, row in gs.iterrows():
    if pd.notna(row['Pct_Population']):
        axes[0].text(i, row['Pct_Population'] * 100 + 0.3,
                     f"{row['Pct_Population']*100:.1f}%",
                     ha='center', fontweight='bold')
axes[0].set_title('Phân bố KH theo Grade', fontsize=11, fontweight='bold')
axes[0].set_ylabel('% Khách hàng')

# Chart 2: Bad rate theo grade
axes[1].bar(gs['Grade'], gs['Bad_Rate'] * 100,
            color=grade_colors, edgecolor='white', linewidth=1.5)
axes[1].axhline(y=y_test.mean() * 100, color='black',
                linestyle='--', linewidth=1.5,
                label=f'Baseline: {y_test.mean()*100:.1f}%')
for i, row in gs.iterrows():
    if pd.notna(row['Bad_Rate']):
        axes[1].text(i, row['Bad_Rate'] * 100 + 0.2,
                     f"{row['Bad_Rate']*100:.1f}%",
                     ha='center', fontweight='bold')
axes[1].set_title('Bad Rate theo Grade\n(phải monotonic: A < B < C < D < E)',
                  fontsize=11, fontweight='bold')
axes[1].set_ylabel('Bad Rate (%)')
axes[1].legend()

# Chart 3: Boxplot theo grade
for i, (g, color) in enumerate(zip(grade_order, grade_colors)):
    subset = grade_analysis[grade_analysis['Grade'] == g]['Score']
    if len(subset) > 0:   # chỉ vẽ nếu có data — tránh NaN error
        axes[2].boxplot(subset, positions=[i], widths=0.6,
                        patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.7),
                        medianprops=dict(color='black', linewidth=2))
axes[2].set_xticks(range(5))
axes[2].set_xticklabels(grade_order)
axes[2].set_title('Score Distribution theo Grade', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Credit Score')

plt.suptitle('Risk Grade Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.5. Bảng score card

### 5.5.1. Bảng điểm Score card cho mỗi feature

In [ ]:
# Tạo bảng scorecard từ WOE bins và hệ số Logistic Regression

coef_df = pd.DataFrame({
    "Feature": X_train_woe_sel.columns,
    "Beta": lr_model.coef_[0]
})

coef_df["Abs_Beta"] = coef_df["Beta"].abs()
coef_df = coef_df.sort_values("Abs_Beta", ascending=False).reset_index(drop=True)

n_features = len(coef_df)

# Base score chia đều cho các feature để bảng scorecard có thể cộng điểm theo từng bin
base_score_total = OFFSET - FACTOR * lr_model.intercept_[0]
base_score_per_feature = base_score_total / n_features

scorecard_rows = []

for _, row in coef_df.iterrows():
    model_feature = row["Feature"]
    beta = row["Beta"]

    if model_feature.endswith("_woe"):
        raw_feature = model_feature.replace("_woe", "")

        if raw_feature not in bins_all:
            continue

        bin_df = bins_all[raw_feature].copy()

        for _, bin_row in bin_df.iterrows():
            woe = bin_row["woe"]

            # Score contribution của feature/bin
            score_points = base_score_per_feature - FACTOR * beta * woe

            scorecard_rows.append({
                "Model_Feature": model_feature,
                "Feature": raw_feature,
                "Bin": bin_row["bin_label"],
                "WOE": woe,
                "Beta": beta,
                "Score_Points": score_points,
            })

    else:
        # Trường hợp feature binary được append trực tiếp, ví dụ *_IS_MISSING
        for value in [0, 1]:
            score_points = base_score_per_feature - FACTOR * beta * value

            scorecard_rows.append({
                "Model_Feature": model_feature,
                "Feature": model_feature,
                "Bin": f"{model_feature} = {value}",
                "WOE": value,
                "Beta": beta,
                "Score_Points": score_points,
            })

scorecard_df = pd.DataFrame(scorecard_rows)

print(f"Số feature trong LR model: {n_features}")
print(f"Base score total: {base_score_total:.2f}")
print(f"Base score per feature: {base_score_per_feature:.2f}")
print(f"Số dòng scorecard: {len(scorecard_df)}")

display(scorecard_df.head(10))


In [ ]:
# Hiển thị scorecard cho 3-5 features quan trọng nhất

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

n_show_features = 5  # đổi thành 3 nếu chỉ muốn xem 3 feature

top_model_features = coef_df.head(n_show_features)["Feature"].tolist()

for model_feature in top_model_features:
    sc = scorecard_df[scorecard_df["Model_Feature"] == model_feature].copy()

    if sc.empty:
        print(f"Không tìm thấy scorecard cho feature: {model_feature}")
        continue

    # Format giống mẫu: Columns | Features | WOE | Betas | Scores
    display_df = pd.DataFrame({
        "Columns": sc["Feature"],
        "Features": sc["Bin"],
        "WOE": sc["WOE"],
        "Betas": sc["Beta"],
        "Scores": sc["Score_Points"],
    }).reset_index(drop=True)

    print("\n" + "=" * 100)
    print(f"SCORECARD: {model_feature}")
    print("=" * 100)

    display(display_df)


### 5.5.2. Bảng thông tin tổng hợp về độ tín nhiệm cho 1 hồ sơ cụ thể
Sau khi tính điểm số mỗi feature, Tính toán tín nhiệm cho một hồ sơ cụ thể

Các thông tin hiển thị: 

**1. Probability of Default (PD)**  
PD là xác suất khách hàng thuộc nhóm BAD theo Logistic Regression.  
Output này phục vụ đánh giá rủi ro định lượng, so sánh model và dùng cho các bài toán như pricing hoặc expected loss nếu model đã được kiểm tra calibration.

**2. Credit Score và Risk Grade**  
- Credit score là phiên bản quy đổi từ PD sang thang điểm dễ đọc hơn. Score cao nghĩa là rủi ro thấp hơn.  
- Risk Grade A/B/C/D/E được gán từ tổng credit score của hồ sơ, phục vụ nhân viên đánh giá tín dụng khách hàng, đặt cutoff, chia nhóm khách hàng và theo dõi bad rate theo từng nhóm.

**3. Feature Contribution**  
Bảng contribution cho biết từng feature đang làm tổng điểm của hồ sơ tăng hay giảm.  
Mỗi dòng thể hiện bin/WOE của feature, hệ số beta của Logistic Regression và số điểm đóng góp vào tổng score.


In [ ]:
# Chọn 1 hồ sơ trong tập test

customer_pos = 2  # chọn hồ sơ thứ 3 trong test set (index 2) để phân tích chi tiết
#customer_pos = np.argmax(y_test_prob) # chọn hồ sơ có PD cao nhất trong test set
#customer_pos = np.argmin(y_test_prob) # chọn hồ sơ có PD thấp nhất trong test set
customer_idx = X_test_woe_sel.index[customer_pos]
customer_features = X_test_woe_sel.loc[customer_idx]

prob_bad = y_test_prob[customer_pos]
score = test_scores[customer_pos]
grade = test_grades.iloc[customer_pos]
actual = y_test.iloc[customer_pos]

print("=" * 90)
print("HỒ SƠ TÍN DỤNG ")
print("=" * 90)
print(f"Customer index : {customer_idx}")
print(f"Actual TARGET  : {actual} ({'BAD' if actual == 1 else 'GOOD'})")
print(f"PD / Prob Bad (XS vỡ nợ)  : {prob_bad:.4%}")
print(f"Credit Score   : {score}")
print(f"Risk Grade     : {grade} - {GRADE_LABELS[grade]}")
print("=" * 90)

# Tính contribution từng feature vào log-odds và score
contrib_rows = []

for feature, beta in zip(X_train_woe_sel.columns, lr_model.coef_[0]):
    value = customer_features[feature]
    log_odds_contribution = beta * value
    score_contribution = -FACTOR * log_odds_contribution

    # Lấy bin/label tương ứng từ scorecard_df nếu có
    sc_feature = scorecard_df[scorecard_df["Model_Feature"] == feature].copy()

    matched_bin = None
    matched_woe = value

    if feature.endswith("_woe"):
        # WOE value đã là giá trị sau transform, match gần đúng theo WOE trong scorecard
        if not sc_feature.empty:
            sc_feature["woe_diff"] = (sc_feature["WOE"] - value).abs()
            best_row = sc_feature.sort_values("woe_diff").iloc[0]
            matched_bin = best_row["Bin"]
            matched_woe = best_row["WOE"]
    else:
        # Binary flag 0/1
        matched_bin = f"{feature} = {int(value)}"

    contrib_rows.append({
        "Feature": feature,
        "Bin": matched_bin,
        "Value_WOE": value,
        "Beta": beta,
        "LogOdds_Contribution": log_odds_contribution,
        "Score_Contribution": score_contribution,
    })

contrib_df = pd.DataFrame(contrib_rows)

# Sắp xếp theo tác động tuyệt đối lên score
contrib_df["Abs_Score_Contribution"] = contrib_df["Score_Contribution"].abs()
contrib_df = contrib_df.sort_values("Abs_Score_Contribution", ascending=False)

display_cols = [
    "Feature",
    "Bin",
    "Value_WOE",
    "Beta",
    "LogOdds_Contribution",
    "Score_Contribution",
]

print("\nTop feature contributions:")
display(contrib_df[display_cols].head(20).round(4))

# Kiểm tra tổng điểm từ contribution
base_score = OFFSET - FACTOR * lr_model.intercept_[0]
score_from_contrib = base_score + contrib_df["Score_Contribution"].sum()

print("\nScore reconciliation:")
print(f"Base score trước khi xét feature : {base_score:.2f}")
print(f"Tổng điểm cộng/ trừ từ all feature  : {contrib_df['Score_Contribution'].sum():.2f}")
print(f"Điểm sau khi có feature          : {score_from_contrib:.2f}")
print(f"Displayed rounded score   : {score}")


## 5.6. Business Cutoff: Từ Score sang Quyết định Approve / Reject

Sau khi model tạo ra `Probability of Default (PD)` và `Credit Score`, bước tiếp theo là chọn một **cutoff** để ra quyết định tín dụng.

Trong bài làm này, dùng ranh giới giữa **Grade C và Grade D** làm cutoff:

- **Approve**: khách hàng thuộc Grade A/B/C, tức là nhóm có score cao hơn cutoff.
- **Reject / Review**: khách hàng thuộc Grade D/E, tức là nhóm có score thấp hơn cutoff.

Ý nghĩa business của cutoff này:

- `Approval Rate`: tỷ lệ hồ sơ được duyệt.
- `Bad Rate in Approved`: trong nhóm được duyệt, bao nhiêu % thực tế là khách BAD.
- `Bad Capture Rate`: trong toàn bộ khách BAD, model chặn được bao nhiêu %.
- `Good Rejection Rate`: trong toàn bộ khách GOOD, bao nhiêu % bị reject nhầm.

Trade-off chính:

- Cutoff càng cao → duyệt ít hơn, bad rate thấp hơn, nhưng reject nhiều khách tốt hơn.
- Cutoff càng thấp → duyệt nhiều hơn, tăng doanh số, nhưng nhận thêm rủi ro bad debt.

Vì vậy cutoff không chỉ là quyết định kỹ thuật, mà là quyết định business phụ thuộc vào khẩu vị rủi ro của tổ chức.


In [ ]:
# Business Cutoff Analysis: Approve / Reject theo ranh giới Grade C/D

from sklearn.metrics import classification_report, confusion_matrix

# Cutoff score: ranh giới Grade C/D
cutoff_score = q40 # có thể là q60 nếu muốn ranh giới giữa B/C

# Đổi cutoff score về PD threshold bằng công thức score scaling:
# score = OFFSET - FACTOR * log_odds_bad
log_odds_cut = (OFFSET - cutoff_score) / FACTOR
pd_cutoff = np.exp(log_odds_cut) / (1 + np.exp(log_odds_cut))

print("BUSINESS CUTOFF")
print(f"Cutoff score       : {cutoff_score}")
print(f"PD threshold       : {pd_cutoff:.4%}")
print("Decision rule      :")
print(f"  - Approve        : score >= {cutoff_score}  (Grade A/B/C)")
print(f"  - Reject/Review  : score <  {cutoff_score}  (Grade D/E)")
print("=" * 90)

# Decision theo score
decision_df = pd.DataFrame({
    "Actual": y_test.values,
    "PD": y_test_prob,
    "Score": test_scores,
    "Grade": test_grades.values,
})

decision_df["Decision"] = np.where(
    decision_df["Score"] >= cutoff_score,
    "APPROVE",
    "REJECT/REVIEW"
)

# y_pred_cut: 1 = predict BAD / reject, 0 = predict GOOD / approve
y_pred_cut = np.where(decision_df["Decision"] == "REJECT/REVIEW", 1, 0)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_cut)
tn, fp, fn, tp = cm.ravel()

total = len(decision_df)
actual_good = (decision_df["Actual"] == 0).sum()
actual_bad = (decision_df["Actual"] == 1).sum()

approved = (decision_df["Decision"] == "APPROVE").sum()
rejected = (decision_df["Decision"] == "REJECT/REVIEW").sum()

approved_bad = ((decision_df["Decision"] == "APPROVE") & (decision_df["Actual"] == 1)).sum()
approved_good = ((decision_df["Decision"] == "APPROVE") & (decision_df["Actual"] == 0)).sum()

rejected_bad = ((decision_df["Decision"] == "REJECT/REVIEW") & (decision_df["Actual"] == 1)).sum()
rejected_good = ((decision_df["Decision"] == "REJECT/REVIEW") & (decision_df["Actual"] == 0)).sum()

# Business metrics
approval_rate = approved / total
reject_rate = rejected / total

bad_rate_approved = approved_bad / approved if approved > 0 else 0
good_rate_approved = approved_good / approved if approved > 0 else 0

bad_capture_rate = rejected_bad / actual_bad if actual_bad > 0 else 0
good_rejection_rate = rejected_good / actual_good if actual_good > 0 else 0

bad_leakage_rate = approved_bad / actual_bad if actual_bad > 0 else 0

# Summary table
business_summary = pd.DataFrame({
    "Metric": [
        "Total applications",
        "Approval rate",
        "Reject / review rate",
        "Bad rate in approved portfolio",
        "Good rate in approved portfolio",
        "Bad capture rate",
        "Bad leakage rate",
        "Good rejection rate",
    ],
    "Value": [
        f"{total:,}",
        f"{approval_rate:.2%}",
        f"{reject_rate:.2%}",
        f"{bad_rate_approved:.2%}",
        f"{good_rate_approved:.2%}",
        f"{bad_capture_rate:.2%}",
        f"{bad_leakage_rate:.2%}",
        f"{good_rejection_rate:.2%}",
    ],
    "Business Meaning": [
        "Tổng số hồ sơ trong tập test/OOT",
        "Tỷ lệ hồ sơ được duyệt",
        "Tỷ lệ hồ sơ bị từ chối hoặc đưa vào manual review",
        "Rủi ro thực tế trong nhóm được duyệt",
        "Tỷ lệ khách tốt trong nhóm được duyệt",
        "Trong toàn bộ khách BAD, model chặn được bao nhiêu %",
        "Trong toàn bộ khách BAD, bao nhiêu % vẫn lọt vào nhóm approve",
        "Trong toàn bộ khách GOOD, bao nhiêu % bị reject nhầm",
    ]
})

display(business_summary)

# Confusion matrix theo ngôn ngữ business
business_cm = pd.DataFrame({
    "Segment": [
        "Approve correctly",
        "Approve wrongly",
        "Reject correctly",
        "Reject wrongly",
    ],
    "Actual": [
        "GOOD",
        "BAD",
        "BAD",
        "GOOD",
    ],
    "Decision": [
        "APPROVE",
        "APPROVE",
        "REJECT/REVIEW",
        "REJECT/REVIEW",
    ],
    "Count": [
        approved_good,
        approved_bad,
        rejected_bad,
        rejected_good,
    ],
    "Business Meaning": [
        "Khách tốt được duyệt đúng",
        "Khách xấu bị duyệt nhầm - credit loss risk",
        "Khách xấu bị chặn đúng",
        "Khách tốt bị reject nhầm - opportunity cost",
    ]
})

display(business_cm)

print("\nClassification Report tại cutoff business:")
print(classification_report(
    y_test,
    y_pred_cut,
    target_names=["GOOD (0)", "BAD (1)"],
    digits=4
))

print("\nBusiness interpretation:")
print(
    f"Với cutoff score = {cutoff_score}, model duyệt {approval_rate:.1%} hồ sơ. "
    f"Bad rate trong nhóm được duyệt là {bad_rate_approved:.2%}. "
    f"Model chặn được {bad_capture_rate:.1%} tổng số khách BAD, "
    f"nhưng cũng reject nhầm {good_rejection_rate:.1%} khách GOOD."
)


# 6. Thử nghiệm xây dựng mô hình LightGBM
LightGBM là Gradient Boosted Decision Trees, khai thác **non-linear relationships
và feature interactions** mà Logistic Regression không bắt được.

**LightGBM không cần WOE transform?**
- Tree-based model tự tìm ngưỡng tốt nhất cho từng split → không cần linearize
- Tự xử lý được missing values (NaN) không cần impute
- Tự tìm interaction giữa các features (điều LR + WOE bỏ qua hoàn toàn)
- Ít nhạy cảm với outliers và skewed distribution

Mục tiêu: Đánh giá với mô hình LR, trade-off giữa GINI/ hiệu năng và  khả năng diễn giải


## 6.1. Baseline LightGBM
Train model  với hyperparameters mặc định hợp lý
để có baseline GINI để so sánh.

In [ ]:
# Baseline LightGBM
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import re

# 1. Tách validation từ training window
# X_tr  : dùng để fit model
# X_val : dùng cho early stopping / model selection
# X_test: OOT test set giữ nguyên để report cuối
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

# 2. Xóa ký tự đặc biệt trong tên cột để LightGBM ổn định
X_tr = X_tr.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))
X_val = X_val.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))
X_test_lgb = X_test.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))

# 3. Ép kiểu biến chữ về Category cho LightGBM
obj_cols = X_tr.select_dtypes(['object']).columns
X_tr[obj_cols] = X_tr[obj_cols].astype('category')
X_val[obj_cols] = X_val[obj_cols].astype('category')
X_test_lgb[obj_cols] = X_test_lgb[obj_cols].astype('category')

# 4. Huấn luyện baseline: early stopping trên validation, report cuối trên OOT test
lgb_baseline = lgb.LGBMClassifier(
    random_state=42,
    class_weight='balanced' # xử lý class imbalance bằng cách tăng trọng số class BAD trong training
)

lgb_baseline.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(100)
    ]
)

val_auc_base = roc_auc_score(
    y_val,
    lgb_baseline.predict_proba(X_val)[:, 1]
)

oot_auc_base = roc_auc_score(
    y_test,
    lgb_baseline.predict_proba(X_test_lgb)[:, 1]
)

print(f"\nBaseline Validation GINI: {2 * val_auc_base - 1:.4f}")
print(f"Baseline OOT Test GINI:   {2 * oot_auc_base - 1:.4f}")


## 6.2. Thử nghiệm finetune Hyperparameter (Config-based)

**3 chiến lược:**
- `baseline`: params mặc định tốt — dùng làm reference, chọn model theo Validation GINI
- `low_complexity`: giảm `num_leaves` từ 31 → 15 — ít cành hơn → ít overfit hơn
- `high_reg`: tăng L1/L2 mạnh + giảm sampling rate — penalty nặng hơn
.


In [ ]:
# Finetuning Configurations — Stability-first model selection

CONFIGS = {
    'baseline': {
        'desc'             : 'Baseline — params mặc định tốt',
        'n_estimators'     : 500,
        'learning_rate'    : 0.05,
        'num_leaves'       : 31,
        'min_child_samples': 50,
        'subsample'        : 0.8,
        'colsample_bytree' : 0.8,
        'reg_alpha'        : 0.1,
        'reg_lambda'       : 1.0,
    },
    'low_complexity': {
        'desc'             : 'Low complexity — ít lá, ổn định hơn',
        'n_estimators'     : 1000,
        'learning_rate'    : 0.03,
        'num_leaves'       : 15,
        'min_child_samples': 100,
        'subsample'        : 0.8,
        'colsample_bytree' : 0.8,
        'reg_alpha'        : 0.1,
        'reg_lambda'       : 1.0,
    },
    'high_reg': {
        'desc'             : 'High regularization — L1/L2 mạnh + sampling thấp',
        'n_estimators'     : 1000,
        'learning_rate'    : 0.03,
        'num_leaves'       : 31,
        'min_child_samples': 100,
        'subsample'        : 0.7,
        'colsample_bytree' : 0.7,
        'reg_alpha'        : 1.0,
        'reg_lambda'       : 10.0,
    },
}

BASE_PARAMS = {
    'objective'       : 'binary',
    'metric'          : 'auc',
    'boosting_type'   : 'gbdt',
    'class_weight'    : 'balanced',
    'subsample_freq'  : 1,
    'random_state'    : 42,
    'n_jobs'          : -1,
    'verbose'         : -1,
}

# Gap = Train GINI - Test/OOT GINI.
MAX_ACCEPTABLE_GAP = 0.18

exp_results = {}

for name, cfg in CONFIGS.items():
    desc = cfg['desc']
    params = {**BASE_PARAMS, **{k: v for k, v in cfg.items() if k != 'desc'}}

    print(f"Chạy Config: [{name}] — {desc}")

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    tr_prob   = model.predict_proba(X_tr)[:, 1]
    val_prob  = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test_lgb)[:, 1]

    tr_gini   = round(2 * roc_auc_score(y_tr, tr_prob) - 1, 4)
    val_gini  = round(2 * roc_auc_score(y_val, val_prob) - 1, 4)
    test_gini_cfg = round(2 * roc_auc_score(y_test, test_prob) - 1, 4)
    gap = round(tr_gini - test_gini_cfg, 4)

    exp_results[name] = {
        'desc'      : desc,
        'Train GINI': tr_gini,
        'Val GINI'  : val_gini,
        'Test GINI' : test_gini_cfg,
        'Gap'       : gap,
        'Stable'    : gap <= MAX_ACCEPTABLE_GAP,
        'Best iter' : model.best_iteration_,
        'model'     : model,
        'te_prob'   : test_prob,
        'val_prob'  : val_prob,
        'tr_prob'   : tr_prob,
    }

    print(
        f"Train GINI: {tr_gini:.4f} | "
        f"Val GINI: {val_gini:.4f} | "
        f"OOT Test GINI: {test_gini_cfg:.4f} | "
        f"Gap: {gap:.4f} | "
        f"Stable: {'Yes' if gap <= MAX_ACCEPTABLE_GAP else 'No'} | "
        f"Best Iter: {model.best_iteration_}"
    )

print("BẢNG TỔNG HỢP")

summary_rows = []
for name, r in exp_results.items():
    summary_rows.append({
        'Config'     : name,
        'Description': r['desc'],
        'Train GINI' : r['Train GINI'],
        'Val GINI'   : r['Val GINI'],
        'Test GINI'  : r['Test GINI'],
        'Gap'        : r['Gap'],
        'Stable'     : r['Stable'],
        'Best iter'  : r['Best iter'],
    })

summary_df = pd.DataFrame(summary_rows)

# Sort để dễ đọc: stable trước, sau đó Val GINI giảm dần
summary_df = summary_df.sort_values(
    ['Stable', 'Val GINI'],
    ascending=[False, False]
).reset_index(drop=True)

display(summary_df)

# Chọn model stability-first:
# 1. Chỉ xét config stable nếu có
# 2. Trong nhóm stable, chọn Val GINI cao nhất
stable_candidates = summary_df[summary_df['Stable'] == True].copy()

if len(stable_candidates) > 0:
    best_row = stable_candidates.sort_values('Val GINI', ascending=False).iloc[0]
    selection_reason = (
        f"Chọn trong nhóm Stable (Gap <= {MAX_ACCEPTABLE_GAP}) "
        "và có Validation GINI cao nhất."
    )
else:
    best_row = summary_df.sort_values('Val GINI', ascending=False).iloc[0]
    selection_reason = (
        "Không có config nào đạt stability threshold, "
        "fallback chọn Validation GINI cao nhất."
    )

best_config = best_row['Config']

print("\n" + "=" * 70)
print("MODEL SELECTION — STABILITY FIRST")
print(selection_reason)
print(f"BEST CONFIG       : {best_config}")
print(f"Description       : {exp_results[best_config]['desc']}")
print(f"Train GINI        : {exp_results[best_config]['Train GINI']:.4f}")
print(f"Validation GINI   : {exp_results[best_config]['Val GINI']:.4f}")
print(f"OOT Test GINI     : {exp_results[best_config]['Test GINI']:.4f}")
print(f"Train-Test Gap    : {exp_results[best_config]['Gap']:.4f}")
print(f"Best iteration    : {exp_results[best_config]['Best iter']}")


**Phân tích kết quả 3 configs**

Trong credit risk, model không chỉ cần GINI cao mà còn cần ổn định. Vì vậy, tiêu chí chọn model:

1. Ưu tiên các config có `Train GINI - Test GINI` thấp.
2. Trong nhóm stable, chọn config có `Validation GINI` cao nhất.
3. Nếu không config nào đạt stability threshold, fallback chọn config có `Validation GINI` cao nhất.



## 6.3. Train Model với config tốt nhất


In [ ]:
# Gán model tốt nhất từ config experiments

# best_config và exp_results đã được tạo ở cell trước
lgb_final = exp_results[best_config]['model']

y_train_prob_lgb = exp_results[best_config]['tr_prob']
y_val_prob_lgb   = exp_results[best_config]['val_prob']
y_oot_prob_lgb   = exp_results[best_config]['te_prob']   # OOT test probability


y_test_prob_lgb = y_oot_prob_lgb

train_gini_lgb = exp_results[best_config]['Train GINI']
val_gini_lgb   = exp_results[best_config]['Val GINI']
oot_gini_lgb   = exp_results[best_config]['Test GINI']

train_auc_lgb = (train_gini_lgb + 1) / 2
val_auc_lgb   = (val_gini_lgb + 1) / 2
oot_auc_lgb   = (oot_gini_lgb + 1) / 2

test_gini_lgb = oot_gini_lgb
test_auc_lgb  = oot_auc_lgb

print(f"lgb_final = config [{best_config}]")
print(f"  Description      : {exp_results[best_config]['desc']}")
print(f"  Train GINI       : {train_gini_lgb:.4f}")
print(f"  Validation GINI  : {val_gini_lgb:.4f}")
print(f"  OOT Test GINI    : {oot_gini_lgb:.4f}")
print(f"  Train-OOT Test Gap    : {train_gini_lgb - oot_gini_lgb:.4f}")
print(f"  Best iteration   : {exp_results[best_config]['Best iter']}")


## 6.4. Đánh giá mô hình LightGBM


In [ ]:
# Metrics LightGBM: Train / Validation / OOT Test
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score, roc_auc_score

# AUC / GINI trên 3 tập
train_auc_lgb = roc_auc_score(y_tr, y_train_prob_lgb)
val_auc_lgb   = roc_auc_score(y_val, y_val_prob_lgb)
oot_auc_lgb   = roc_auc_score(y_test, y_test_prob_lgb)

train_gini_lgb = 2 * train_auc_lgb - 1
val_gini_lgb   = 2 * val_auc_lgb - 1
test_gini_lgb  = 2 * oot_auc_lgb - 1
test_auc_lgb   = oot_auc_lgb

# KS trên OOT test
fpr_lgb, tpr_lgb, thresh_lgb = roc_curve(y_test, y_test_prob_lgb)
ks_lgb = (tpr_lgb - fpr_lgb).max()
ks_lgb_threshold = thresh_lgb[(tpr_lgb - fpr_lgb).argmax()]

# Average Precision trên OOT test
ap_lgb = average_precision_score(y_test, y_test_prob_lgb)

print("LightGBM METRICS")
print("=" * 60)
print(f"Selected config   : {best_config}")
print(f"Train AUC / GINI  : {train_auc_lgb:.4f} / {train_gini_lgb:.4f}")
print(f"Val AUC / GINI    : {val_auc_lgb:.4f} / {val_gini_lgb:.4f}")
print(f"OOT Test AUC / GINI    : {test_auc_lgb:.4f} / {test_gini_lgb:.4f}")
print(f"Train-OOT Gap     : {train_gini_lgb - test_gini_lgb:.4f}")
print(f"OOT KS Statistic  : {ks_lgb:.4f}")
print(f"OOT KS Threshold  : {ks_lgb_threshold:.4f}")
print(f"OOT Avg Precision : {ap_lgb:.4f}")

print("\nSo sánh với LR Scorecard trên cùng OOT test:")
print(f"  LR OOT GINI     : {test_gini:.4f}")
print(f"  LGB OOT GINI    : {test_gini_lgb:.4f}")
print(f"  GINI uplift     : +{(test_gini_lgb - test_gini):.4f} ({(test_gini_lgb - test_gini)*100:.2f} điểm %)")


In [ ]:
# Visualize: ROC + PR Curve so sánh 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve — cả 2 track
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_prob)
axes[0].plot(fpr_lr, tpr_lr, color='#1565C0', linewidth=2,
             linestyle='--', label=f'LR (GINI={test_gini:.4f})')
axes[0].plot(fpr_lgb, tpr_lgb, color='#E53935', linewidth=2.5,
             label=f'Track 2 LGB (GINI={test_gini_lgb:.4f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1, alpha=0.5, label='Random')
axes[0].fill_between(fpr_lgb, tpr_lgb, alpha=0.08, color='#E53935')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — LR vs LightGBM', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# PR Curve — cả 2 track
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_test_prob)
prec_lgb, rec_lgb, _ = precision_recall_curve(y_test, y_test_prob_lgb)
ap_lr  = average_precision_score(y_test, y_test_prob)
ap_lgb = average_precision_score(y_test, y_test_prob_lgb)

axes[1].plot(rec_lr,  prec_lr,  color='#1565C0', linewidth=2,
             linestyle='--', label=f'LR (AP={ap_lr:.4f})')
axes[1].plot(rec_lgb, prec_lgb, color='#E53935', linewidth=2.5,
             label=f'LGB (AP={ap_lgb:.4f})')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--',
                linewidth=1, label=f'Baseline={y_test.mean():.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('LR vs LightGBM — Discrimination Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.5. Feature Importance
LightGBM cung cấp 2 loại feature importance:
- `split`: số lần feature được dùng để split — phổ biến hơn, dễ bị bias bởi cardinality
- `gain`: tổng information gain mà feature mang lại — reliable hơn

-> Dùng `gain` để ranking features.


In [ ]:
# Lấy importance theo gain
fi_df = pd.DataFrame({
    "Feature": X_tr.columns,
    "Importance": lgb_final.booster_.feature_importance(importance_type="gain"),
}).sort_values("Importance", ascending=False).reset_index(drop=True)

# So sánh với IV từ model LR
fi_df_merged = fi_df.merge(
    iv_df_clean[["Variable", "IV"]].rename(columns={"Variable": "Feature"}),
    on="Feature",
    how="left"
)

print("Top 20 features theo LightGBM:")

print(fi_df_merged.head(20)[["Feature", "Importance", "IV"]].to_string(index=False))

**Nhận xét Feature Importance — LightGBM**

Top features của LightGBM cho thấy model đang học từ các nhóm biến hợp lý về mặt nghiệp vụ:

1. **External credit scores là nhóm signal mạnh nhất**
   - `EXT_SOURCE_MEAN`, `EXT_SOURCE_MAX`, `EXT_SOURCE_MIN`, `EXT_SOURCE_3`, `EXT_SOURCE_1`, `EXT_SOURCE_PRODUCT` đều xuất hiện trong top 20.
   - Đây là các biến tổng hợp điểm tín dụng bên ngoài.
   - `EXT_SOURCE_MEAN` có importance cao nhất và IV cũng rất cao (`IV ≈ 0.60`), cho thấy biến này nhất quán giữa cả LightGBM và WOE/IV.

2. **Biến tài chính và khả năng trả nợ có đóng góp rõ**
   - `ANNUITY_TO_CREDIT`, `CREDIT_TO_GOODS`, `AMT_ANNUITY`, `AMT_GOODS_PRICE` xuất hiện trong top 20.
   - Đây là các biến phản ánh cấu trúc khoản vay, quy mô khoản vay và gánh nặng trả nợ.
   - Các biến này có IV từ weak đến medium, nhưng LightGBM vẫn khai thác tốt nhờ khả năng học non-linear split.

3. **Lịch sử tín dụng và hành vi trả nợ là signal quan trọng**
   - `INST_LATE_RATE`, `BUREAU_DEBT_RATIO`, `BUREAU_DAYS_CREDIT_MAX`, `PREV_CREDIT_TO_APP_RATIO`, `PREV_DAYS_DECISION_MEAN`, `PREV_AMT_DOWN_PAYMENT_MEAN` đều nằm trong top 20.
   - Điều này phù hợp với logic credit risk: lịch sử trả chậm, dư nợ, hành vi vay trước đó và khoản trả trước đều liên quan trực tiếp đến rủi ro default.

4. **Categorical variables được LightGBM khai thác tốt**
   - `ORGANIZATION_TYPE`, `OCCUPATION_TYPE`, `NAME_EDUCATION_TYPE`, `CODE_GENDER` nằm trong top 20.
   - `ORGANIZATION_TYPE` và `OCCUPATION_TYPE` có IV = 0 trong bảng do cách tính IV/WOE hiện tại không capture tốt categorical high-cardinality hoặc biến bị exclude/khó binning.
   - Tuy nhiên LightGBM xử lý categorical/raw features tốt hơn, nên vẫn tìm được signal từ các biến này.

5. **So sánh IV và LightGBM importance**
   - Các biến như `EXT_SOURCE_MEAN`, `EXT_SOURCE_MAX`, `EXT_SOURCE_MIN` có cả IV cao và gain cao → signal mạnh, ổn định giữa hai mô hình.
   - Các biến như `ORGANIZATION_TYPE`, `OCCUPATION_TYPE` có gain cao nhưng IV thấp/0 → LightGBM đang khai thác pattern mà LR + WOE không tận dụng tốt.

**Kết luận:** LightGBM không chỉ cải thiện performance nhờ thuật toán mạnh hơn, mà còn khai thác được thêm signal từ categorical variables, interaction và non-linear relationship mà LR Scorecard khó nắm bắt.


In [ ]:
# Visualize Feature Importance

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 theo LightGBM importance
top20 = fi_df.head(20)
axes[0].barh(top20['Feature'][::-1], top20['Importance'][::-1],
             color='#E53935', alpha=0.8, edgecolor='white')
axes[0].set_title('Top 20 — LightGBM Feature Importance (gain)',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Importance (gain)')

# Scatter: LightGBM Importance vs IV (features có cả 2)
fi_both = fi_df_merged.dropna(subset=['IV'])
axes[1].scatter(fi_both['IV'], fi_both['Importance'],
                alpha=0.6, color='#E53935', s=60)

# Annotate top 10
for _, row in fi_both.head(10).iterrows():
    axes[1].annotate(row['Feature'][:20],
                     (row['IV'], row['Importance']),
                     fontsize=7, alpha=0.8)

axes[1].set_xlabel('IV (từ  WOE analysis)')
axes[1].set_ylabel('LightGBM Importance')
axes[1].set_title('IV vs LightGBM Importance\n(tương quan cao: 2 mô hình nhất quán)',
                  fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.suptitle('LightGBM — Feature Importance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.6. So sánh 2 mô hình LR và LightGBM


In [ ]:
# Bảng so sánh toàn diện

from sklearn.metrics import (roc_curve, precision_recall_curve,
                             average_precision_score)

# KS Track 1
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_prob)
ks_lr = (tpr_lr - fpr_lr).max()

# AP scores
ap_lr  = average_precision_score(y_test, y_test_prob)
ap_lgb = average_precision_score(y_test, y_test_prob_lgb)

print("=" * 60)
print("SO SÁNH LR vs LightGBM")
print("=" * 60)

comparison = {
    'Metric'          : ['Train GINI', 'Test/OOT GINI', 'KS Statistic',
                         'AUC', 'Avg Precision', 'Features used',
                         'Benchmark 0.49'],
    'LR'   : [f'{train_gini:.4f}', f'{test_gini:.4f}',
                         f'{ks_lr:.4f}', f'{test_auc:.4f}',
                         f'{ap_lr:.4f}', f'{len(selected_features)} WOE/binary features',
                         'Đạt' if test_gini >= 0.49 else 'Chưa đạt'],
    'LGB'  : [f'{train_gini_lgb:.4f}', f'{test_gini_lgb:.4f}',
                         f'{ks_lgb:.4f}', f'{test_auc_lgb:.4f}',
                         f'{ap_lgb:.4f}', f'{X_tr.shape[1]} raw features',
                         'Đạt' if test_gini_lgb >= 0.49 else 'Chưa đạt'],
}

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

delta = test_gini_lgb - test_gini
print(f"\nCải thiện GINI: +{delta:.4f} ({delta*100:.2f}% points)")


Trade off giữa 2 mô hình:

**LR Scorecard**

- Dễ giải thích, phù hợp với credit policy, audit và regulatory review.
- Có thể chuyển thành bảng điểm từng bin.
- Probability/score dễ diễn giải hơn nếu không dùng class weighting.
- Hiệu năng thường thấp hơn vì chỉ học quan hệ tuyến tính trên WOE features.

**LightGBM**

- Khai thác non-linear relationship và feature interaction tốt hơn.
- Thường đạt GINI/KS cao hơn.
- Phù hợp cho challenger model hoặc auto-decisioning khi cần performance.
- Cần thêm SHAP, calibration và monitoring nếu dùng trong production.

# 7. Tổng kết


## 7.1. Dashboard tổng hợp


In [ ]:
# Dashboard so sánh LR Scorecard vs LightGBM trên OOT test
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

BENCHMARK_GINI = 0.49
GAP_THRESHOLD = globals().get("MAX_ACCEPTABLE_GAP", 0.18)

COLORS = {
    "LR": "#1565C0",
    "LGB": "#C62828",
    "Benchmark": "#9E9E9E",
    "Threshold": "#FF6F00"
}

def add_bar_labels(ax, values, offset=0.005):
    for i, val in enumerate(values):
        ax.text(
            i, val + offset, f"{val:.4f}",
            ha="center", va="bottom",
            fontweight="bold", fontsize=9
        )

# Metrics
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_test_prob)
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, y_test_prob_lgb)

prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_test_prob)
prec_lgb, rec_lgb, _ = precision_recall_curve(y_test, y_test_prob_lgb)

ks_lr = (tpr_lr - fpr_lr).max()
ks_lgb = (tpr_lgb - fpr_lgb).max()

ap_lr = average_precision_score(y_test, y_test_prob)
ap_lgb = average_precision_score(y_test, y_test_prob_lgb)

gap_lr = train_gini - test_gini
gap_lgb = train_gini_lgb - test_gini_lgb

lr_status = "Ổn định" if gap_lr <= GAP_THRESHOLD else "Cần monitor"
lgb_status = "Ổn định" if gap_lgb <= GAP_THRESHOLD else "Cần monitor"

# Layout
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor("#F8F9FA")
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.35)

ax_roc = fig.add_subplot(gs[0, :2])
ax_pr = fig.add_subplot(gs[1, :2])
ax_gini = fig.add_subplot(gs[0, 2])
ax_ks = fig.add_subplot(gs[1, 2])
ax_gap = fig.add_subplot(gs[2, 0])
ax_ap = fig.add_subplot(gs[2, 1])
ax_text = fig.add_subplot(gs[2, 2])

# ROC
ax_roc.plot(fpr_lr, tpr_lr, "--", color=COLORS["LR"], linewidth=2,
            label=f"LR Scorecard (GINI={test_gini:.4f})")
ax_roc.plot(fpr_lgb, tpr_lgb, color=COLORS["LGB"], linewidth=2.5,
            label=f"LightGBM (GINI={test_gini_lgb:.4f})")
ax_roc.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.4)
ax_roc.set_title("ROC Curve - OOT Test", fontweight="bold")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(fontsize=9)
ax_roc.grid(alpha=0.2)

# Precision-Recall
bad_rate = y_test.mean()
ax_pr.plot(rec_lr, prec_lr, "--", color=COLORS["LR"], linewidth=2,
           label=f"LR Scorecard (AP={ap_lr:.4f})")
ax_pr.plot(rec_lgb, prec_lgb, color=COLORS["LGB"], linewidth=2.5,
           label=f"LightGBM (AP={ap_lgb:.4f})")
ax_pr.axhline(bad_rate, color="gray", linestyle=":", linewidth=1.5,
              label=f"Baseline bad rate={bad_rate:.3f}")
ax_pr.set_title("Precision-Recall Curve - OOT Test", fontweight="bold")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend(fontsize=9)
ax_pr.grid(alpha=0.2)

# OOT GINI
gini_values = [test_gini, test_gini_lgb, BENCHMARK_GINI]
ax_gini.bar(["LR", "LGB", "Benchmark"], gini_values,
            color=[COLORS["LR"], COLORS["LGB"], COLORS["Benchmark"]],
            edgecolor="white", width=0.55)
ax_gini.axhline(BENCHMARK_GINI, color=COLORS["Threshold"],
                linestyle="--", linewidth=1.5)
add_bar_labels(ax_gini, gini_values)
ax_gini.set_ylim(0.4, max(gini_values) + 0.05)
ax_gini.set_title("OOT Test GINI", fontweight="bold")
ax_gini.set_ylabel("GINI")

# KS
ks_values = [ks_lr, ks_lgb]
ax_ks.bar(["LR", "LGB"], ks_values,
          color=[COLORS["LR"], COLORS["LGB"]],
          edgecolor="white", width=0.55)
ax_ks.axhline(0.30, color=COLORS["Threshold"], linestyle="--",
              linewidth=1.5, label="KS > 0.30")
add_bar_labels(ax_ks, ks_values)
ax_ks.set_ylim(0, max(ks_values) + 0.08)
ax_ks.set_title("OOT KS Statistic", fontweight="bold")
ax_ks.legend(fontsize=8)

# Train-OOT Gap
gap_values = [gap_lr, gap_lgb]
ax_gap.bar(["LR", "LGB"], gap_values,
           color=["#42A5F5", "#EF9A9A"],
           edgecolor="white", width=0.55)
ax_gap.axhline(GAP_THRESHOLD, color=COLORS["Threshold"],
               linestyle="--", linewidth=1.5,
               label=f"Gap <= {GAP_THRESHOLD:.2f}")
add_bar_labels(ax_gap, gap_values, offset=0.003)
ax_gap.set_ylim(0, max(gap_values + [GAP_THRESHOLD]) + 0.05)
ax_gap.set_title("Train-OOT Gap", fontweight="bold")
ax_gap.legend(fontsize=8)

# Average Precision
ap_values = [ap_lr, ap_lgb]
ax_ap.bar(["LR", "LGB"], ap_values,
          color=[COLORS["LR"], COLORS["LGB"]],
          edgecolor="white", width=0.55)
ax_ap.axhline(bad_rate, color="gray", linestyle="--",
              linewidth=1.5, label="Bad rate baseline")
add_bar_labels(ax_ap, ap_values, offset=0.002)
ax_ap.set_ylim(0, max(ap_values + [bad_rate]) + 0.05)
ax_ap.set_title("Average Precision - PR AUC", fontweight="bold")
ax_ap.legend(fontsize=8)

# Summary
ax_text.axis("off")
summary_text = (
    "FINAL REPORT - OOT TEST\n\n"
    f"LR Scorecard\n"
    f"  GINI : {test_gini:.4f}\n"
    f"  KS   : {ks_lr:.4f}\n"
    f"  Gap  : {gap_lr:.4f} ({lr_status})\n\n"
    f"LightGBM\n"
    f"  GINI : {test_gini_lgb:.4f}\n"
    f"  KS   : {ks_lgb:.4f}\n"
    f"  Gap  : {gap_lgb:.4f} ({lgb_status})\n\n"
    f"Benchmark GINI: {BENCHMARK_GINI:.4f}\n"
    f"LR  : {'Đạt' if test_gini >= BENCHMARK_GINI else 'Chưa đạt'}\n"
    f"LGB : {'Đạt' if test_gini_lgb >= BENCHMARK_GINI else 'Chưa đạt'}\n\n"
    f"Uplift LGB vs LR\n"
    f"  GINI: {(test_gini_lgb - test_gini):+.4f}\n"
    f"  KS  : {(ks_lgb - ks_lr):+.4f}"
)

ax_text.text(
    0.05, 0.95, summary_text,
    transform=ax_text.transAxes,
    fontsize=10,
    verticalalignment="top",
    fontfamily="monospace",
    bbox=dict(boxstyle="round", facecolor="#EEF2FF", alpha=0.85)
)

fig.suptitle(
    "Home Credit Default Risk - Model Comparison Dashboard",
    fontsize=16, fontweight="bold", y=0.98
)

plt.savefig("model_comparison_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

print("Đã lưu: model_comparison_dashboard.png")

## 7.2 — Bảng Metrics Toàn Diện


In [ ]:
# Bảng so sánh metrics đầy đủ: LR Scorecard vs LightGBM

BENCHMARK_GINI = globals().get("BENCHMARK_GINI", 0.49)
GAP_THRESHOLD = globals().get("MAX_ACCEPTABLE_GAP", 0.18)

gap_lr = train_gini - test_gini
gap_lgb = train_gini_lgb - test_gini_lgb

lr_stability = "Ổn định" if gap_lr <= GAP_THRESHOLD else "Cần monitor"
lgb_stability = "Ổn định" if gap_lgb <= GAP_THRESHOLD else "Cần monitor"

val_gini_lgb_display = (
    f"{val_gini_lgb:.4f}"
    if "val_gini_lgb" in globals()
    else "N/A"
)

final_table = pd.DataFrame({
    "Metric": [
        "Train GINI",
        "Validation GINI",
        "OOT Test GINI",
        "Train-OTT Gap",
        "Stability",
        "KS Statistic - OOT Test",
        "AUC ROC - OOT Test",
        "Average Precision - OOT Test",
        "Features input",
        "Preprocessing required",
        "Interpretability",
        f"Benchmark GINI >= {BENCHMARK_GINI:.2f}",
    ],
    "LR Scorecard": [
        f"{train_gini:.4f}",
        "N/A",
        f"{test_gini:.4f}",
        f"{gap_lr:.4f}",
        lr_stability,
        f"{ks_lr:.4f}",
        f"{test_auc:.4f}",
        f"{ap_lr:.4f}",
        f"{len(selected_features)} WOE/binary features",
        "WOE binning + Lasso/VIF + score scaling",
        "Cao - giải thích được từng feature/bin",
        "Đạt yêu cầu" if test_gini >= BENCHMARK_GINI else "Chưa đạt",
    ],
    "LightGBM": [
        f"{train_gini_lgb:.4f}",
        val_gini_lgb_display,
        f"{test_gini_lgb:.4f}",
        f"{gap_lgb:.4f}",
        lgb_stability,
        f"{ks_lgb:.4f}",
        f"{test_auc_lgb:.4f}",
        f"{ap_lgb:.4f}",
        f"{X_tr.shape[1]} raw features",
        "Minimal preprocessing + categorical handling",
        "Thấp hơn - cần SHAP/calibration để dùng production",
        "Đạt yêu cầu" if test_gini_lgb >= BENCHMARK_GINI else "Chưa đạt",
    ],
})

display(final_table)

## 7.3. Calibration


Calibration kiểm tra xem xác suất dự báo của model có gần với tỷ lệ default thực tế hay không.

Trong credit risk, có hai khía cạnh cần tách riêng:

- **Ranking performance**: model có xếp khách hàng rủi ro cao lên trên khách hàng rủi ro thấp không. Chỉ số thường dùng là AUC, GINI, KS.
- **Probability accuracy**: nếu model dự báo PD = 10%, thì trong nhóm khách hàng đó tỷ lệ default thực tế có gần 10% không. Đây là phần calibration.

LightGBM thường cho ranking tốt, nhưng raw probability có thể chưa phải là PD đáng tin cậy. Lý do là tree-based boosting chủ yếu tối ưu khả năng tách nhóm, ngoài ra trong bài này model còn dùng `class_weight='balanced'` để xử lý mất cân bằng lớp. Việc re-weight class giúp model học nhóm BAD tốt hơn, nhưng xác suất raw sau model có thể bị lệch so với bad rate thực tế.

=> thử calibration LightGBM bằng Isotonic Regression:

1. Fit calibration layer trên validation set.
2. Dự báo calibrated probability trên OOT test.
3. So sánh trước/sau calibration bằng Brier Score.
4. Kiểm tra GINI có bị giảm nhiều không.

Brier Score càng thấp càng tốt. Nếu sau calibration Brier giảm trong khi GINI gần như giữ nguyên, nghĩa là model vẫn giữ được khả năng xếp hạng rủi ro nhưng xác suất PD trở nên hợp lý hơn.

In [ ]:
# Calibration LightGBM bằng Isotonic Regression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

# Lưu ý: X_val đã được rename và convert category ở phần LightGBM.
# Validation set được dùng để fit calibration layer.
lgb_calibrated = CalibratedClassifierCV(
    estimator=lgb_final,
    method="isotonic",
    cv="prefit"
)

lgb_calibrated.fit(X_val, y_val)

# Predict calibrated probability trên OOT test
y_test_prob_cal = lgb_calibrated.predict_proba(X_test_lgb)[:, 1]

# Metrics sau calibration
cal_auc = roc_auc_score(y_test, y_test_prob_cal)
cal_gini = 2 * cal_auc - 1

brier_lr = brier_score_loss(y_test, y_test_prob)
brier_lgb_raw = brier_score_loss(y_test, y_test_prob_lgb)
brier_lgb_cal = brier_score_loss(y_test, y_test_prob_cal)

calibration_table = pd.DataFrame({
    "Model": [
        "LR Scorecard",
        "LightGBM raw",
        "LightGBM calibrated"
    ],
    "OOT AUC": [
        test_auc,
        test_auc_lgb,
        cal_auc
    ],
    "OOT GINI": [
        test_gini,
        test_gini_lgb,
        cal_gini
    ],
    "Brier Score": [
        brier_lr,
        brier_lgb_raw,
        brier_lgb_cal
    ]
})

display(calibration_table)

print("Calibration summary")
print(f"LightGBM raw        : GINI = {test_gini_lgb:.4f} | Brier = {brier_lgb_raw:.4f}")
print(f"LightGBM calibrated : GINI = {cal_gini:.4f} | Brier = {brier_lgb_cal:.4f}")
print(f"Brier improvement   : {brier_lgb_raw - brier_lgb_cal:+.4f}")
print("\nValidation set được dùng để fit calibration layer.")
print("OOT test chỉ được dùng để report sau calibration.")


**Nhận xét kết quả Calibration**

Sau calibration, GINI của LightGBM gần như không thay đổi: từ `0.5602` xuống `0.5597`. Mức giảm rất nhỏ, cho thấy calibration không làm mất đáng kể khả năng xếp hạng rủi ro của model.

Brier Score cải thiện rõ rệt, từ `0.1955` xuống `0.0718`, giảm `0.1237`. Điều này cho thấy raw probability của LightGBM ban đầu chưa phản ánh tốt xác suất default thực tế, và Isotonic Regression đã điều chỉnh xác suất về mức hợp lý hơn.

Diễn giải:

- LightGBM raw vẫn xếp hạng khách hàng tốt, thể hiện qua GINI cao.
- Tuy nhiên, xác suất PD raw chưa nên dùng trực tiếp cho pricing, expected loss hoặc provisioning.
- Sau calibration, PD đáng tin cậy hơn trong khi ranking performance gần như được giữ nguyên.

In [ ]:
# Calibration curve + decile table
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))

for name, prob, color in [
    ("LR Scorecard", y_test_prob, "#1565C0"),
    ("LightGBM raw", y_test_prob_lgb, "#C62828"),
    ("LightGBM calibrated", y_test_prob_cal, "#2E7D32"),
]:
    frac_pos, mean_pred = calibration_curve(
        y_test,
        prob,
        n_bins=10,
        strategy="quantile"
    )
    ax.plot(mean_pred, frac_pos, marker="o", linewidth=2, label=name, color=color)

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
ax.set_xlabel("Mean predicted PD")
ax.set_ylabel("Actual bad rate")
ax.set_title("Calibration Curve - OOT Test")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


# Decile calibration table cho LightGBM calibrated
calib_decile_df = pd.DataFrame({
    "y_true": y_test.values,
    "pd_raw": y_test_prob_lgb,
    "pd_calibrated": y_test_prob_cal
})

calib_decile_df["pd_decile"] = pd.qcut(
    calib_decile_df["pd_calibrated"],
    q=10,
    labels=False,
    duplicates="drop"
)

calib_table = (
    calib_decile_df
    .groupby("pd_decile")
    .agg(
        Count=("y_true", "count"),
        Actual_Bad_Rate=("y_true", "mean"),
        Mean_PD_Raw=("pd_raw", "mean"),
        Mean_PD_Calibrated=("pd_calibrated", "mean")
    )
    .reset_index()
)

calib_table["PD_Gap_Calibrated"] = (
    calib_table["Mean_PD_Calibrated"] - calib_table["Actual_Bad_Rate"]
)

display(calib_table)


Calibration curve cho thấy xác suất sau calibration nằm gần đường diagonal hơn so với LightGBM raw. Điều này phù hợp với Brier Score đã cải thiện mạnh sau calibration.

Bảng decile giúp kiểm tra chi tiết hơn: trong từng nhóm PD, so sánh `Mean_PD_Calibrated` với `Actual_Bad_Rate`. Nếu hai giá trị gần nhau, PD của model có thể dùng tốt hơn cho các bài toán cần xác suất như pricing hoặc expected loss.


# Tổng kết bài làm

Bài toán được xây dựng theo hướng credit risk modeling cho Home Credit Default Risk, với mục tiêu dự báo xác suất khách hàng vỡ nợ và chuyển kết quả mô hình thành các output có thể dùng trong nghiệp vụ tín dụng.

Quy trình thực hiện gồm 2 track chính:

- **Track 1: Logistic Regression + WOE/IV + Scorecard**
- **Track 2: LightGBM**

## 1. EDA và tiền xử lý

Dữ liệu được phân tích theo các nhóm chính: biến mục tiêu, missing value, biến numeric, categorical, external credit scores, biến thời gian và biến tài chính.

Một số xử lý chính:

- Chuẩn hóa các giá trị bất thường như `DAYS_EMPLOYED = 365243`.
- Chuyển các biến thời gian từ ngày sang năm để dễ diễn giải.
- Tạo missing flags cho các biến có missing mang ý nghĩa nghiệp vụ.
- Tách train/test theo stratify để giữ tỷ lệ bad rate ổn định.
- Impute missing value dựa trên train set để tránh data leakage.
- Không winsorize outlier trước WOE vì WOE binning đã xử lý biến numeric theo khoảng.

## 2. Feature Engineering

Feature engineering được thực hiện trên cả bảng chính và các bảng phụ.

Từ bảng chính, tạo thêm các biến tỷ lệ tài chính và tổng hợp external scores như:

- `CREDIT_TO_INCOME`
- `ANNUITY_TO_INCOME`
- `ANNUITY_TO_CREDIT`
- `EXT_SOURCE_MEAN`
- `EXT_SOURCE_MIN`
- `EXT_SOURCE_MAX`
- `EXT_SOURCE_PRODUCT`

Từ các bảng phụ như `bureau`, `previous_application`, `POS_CASH_balance`, `installments_payments`, `credit_card_balance`, các feature được aggregate theo `SK_ID_CURR` để phản ánh lịch sử tín dụng, hành vi trả nợ, trạng thái khoản vay và mức độ trễ hạn.

## 3. WOE/IV và Logistic Regression Scorecard

Track Logistic Regression được xây dựng theo hướng scorecard truyền thống:

- Tính IV cho toàn bộ features.
- Chọn biến theo IV.
- Biến đổi các biến sang WOE.
- Loại bớt biến bằng Lasso và VIF để giảm đa cộng tuyến.
- Train Logistic Regression trên tập WOE.
- Chuyển xác suất dự báo thành credit score theo thang điểm 300-850.
- Gán risk grade A/B/C/D/E.
- Xây dựng bảng scorecard cho từng feature/bin.
- Minh họa cách tính điểm cho một hồ sơ cụ thể.

Track này có ưu điểm lớn về khả năng giải thích, phù hợp với môi trường cần audit, policy rule, manual review và giải thích quyết định tín dụng.

## 4. Business Cutoff

Sau khi có credit score và risk grade, xây dựng cutoff nghiệp vụ dựa trên ranh giới Grade C/D.

Ý nghĩa:

- Khách hàng Grade A/B/C được xem là nhóm có thể approve.
- Khách hàng Grade D/E được xem là nhóm nên reject hoặc cần review kỹ hơn.
- Báo cáo thêm approval rate, bad rate trong nhóm approved và bad capture rate để đánh giá tác động kinh doanh của cutoff.

## 5. LightGBM

Track LightGBM được xây dựng như một mô hình challenger với mục tiêu cải thiện khả năng phân biệt rủi ro.

Quy trình gồm:

- Tách thêm validation set từ train để dùng cho early stopping và model selection.
- Giữ OOT test set riêng để đánh giá cuối.
- Xử lý categorical variables theo định dạng phù hợp với LightGBM.
- Chạy baseline model.
- Thử nhiều cấu hình hyperparameter.
- Chọn model theo hướng stability-first: ưu tiên Train-OOT gap ổn định, sau đó mới xét Validation GINI.

Cách chọn này phù hợp hơn với tư duy production vì không chỉ chạy theo GINI cao nhất mà còn kiểm soát overfitting.

## 6. So sánh mô hình

Kết quả cuối so sánh LR Scorecard và LightGBM trên cùng OOT test set qua các chỉ số:

- GINI
- AUC
- KS
- Average Precision
- Train-OOT Gap
- Benchmark GINI 0.49

LightGBM thường cho hiệu năng phân biệt rủi ro tốt hơn, trong khi LR Scorecard có lợi thế rõ ràng về khả năng giải thích và triển khai trong quy trình tín dụng truyền thống.

## 7. Calibration

LightGBM được calibration bằng isotonic regression để cải thiện chất lượng xác suất dự báo.

GINI/KS phản ánh khả năng xếp hạng rủi ro, còn calibration quan trọng khi xác suất PD được dùng cho pricing, expected loss hoặc provisioning.

## 8. Kết luận

LR Scorecard là lựa chọn phù hợp cho baseline production nhờ tính minh bạch, dễ audit và dễ chuyển thành điểm tín dụng/risk grade.

LightGBM phù hợp làm challenger model hoặc dùng trong auto-decisioning nếu có thêm các lớp kiểm soát như SHAP explanation, calibration, monitoring drift và validation định kỳ.

Khuyến nghị triển khai:

- Dùng LR Scorecard cho policy, audit, manual review và giải thích quyết định.
- Dùng LightGBM làm challenger model để cải thiện ranking performance.
- Nếu đưa LightGBM vào production, cần theo dõi PSI, OOT GINI/KS, calibration drift, approval rate và bad rate trong nhóm approved.
